# speech2face -- Emotion LoRA for the Streaming Student (try19/niloys)

Adds **emotion conditioning** to the *production streaming avatar* -- the
try19 **CausalMotionStudent** (38.8M) + **renderer_narrow** stack -- using
targeted LoRA fine-tuning, the same methodology that worked for the
FMGenerator pipeline (`train_emotion_lora_fm_generator_backup.ipynb`).

After training you can make the **live** avatar express **neutral, happy,
sad, angry, surprise or fear**, switchable *at runtime* through a control
file -- without restarting the server.

## Why this is a different notebook

The deployed pipeline no longer runs the FMGenerator at inference time. It runs:

```
PersonaPlex 7B "Helium" hidden states (4096-d @ 12.5 Hz, transformer layer -2)
    -> CausalMotionStudent  (windowed causal transformer, KV cache, 2 frames/step)
    -> raw 32-d motion @ 25 fps
    -> renderer_narrow      (distilled single-identity decoder -> 512x512 frames)
```

| Component | FM version (old notebook) | Student version (this notebook) |
|---|---|---|
| Model trained | `FMGenerator` (~38.8M) | `CausalMotionStudent` (~38.8M, `d_model=768, 6 layers`) |
| Input features | 768-d wav2vec2 (`audio/*.npy`, precomputed) | 4096-d Helium (`helium/*.npy`, extracted here) |
| Emotion injection | `EmotionConditioner` bias in the FMT conditioning sum | Constant bias added to the student `_embed()` output |
| "No emotion" path | CFG null-emotion token | Null anchor: distilled back to the frozen base student |
| LoRA targets | `fmt.c_embedder`, `adaLN_modulation`, `fmt.decoder` | `attn.qkv/proj`, MLPs, `head_abs/head_res` |
| Test renders | FM sampler + full IMTalker renderer | student step loop + renderer_narrow (the live path) |
| Deployment | load checkpoint + set `data["emotion_id"]` | launch `imtalker_try19_student_live_emotion.py`, `echo happy > /workspace/emotion.txt` |

The **32-dim motion space is shared** by the FMGenerator, the student and the
distilled renderer (the student was distilled from the FMGenerator teacher), so
the emotion dataset's `motion/*.pt` files are reused **as-is** -- only the
audio side changes.

**You must provide raw audio** for each dataset clip (the dataset's 768-d
wav2vec2 `audio/*.npy` cannot be converted to 4096-d Helium space). The
notebook extracts `helium/*.npy` (12.5 Hz, 4096-d) from the raw wavs with the
**exact recipe the live server uses** (Mimi -> first 8 codebooks -> LM
embedding sum -> LM transformer, hooked at layer -2).

## What this notebook does

1. clones the emotion-adapter repository (`speech2face_orin_n_original_emotion_adapter_v2`)
2. installs dependencies and detects the GPU (VRAM-based batch size, TF32)
3. writes the student emotion code into the cloned repo
   (`student/emotion_adapter.py`, `student/emotion_dataset.py`,
   `student/extract_helium.py`, `student/train_emotion_student.py`,
   `student/test_emotions_student.py`,
   `live/imtalker_try19_student_live_emotion.py`)
4. downloads the streaming checkpoints (`motion_student_38M.pt`,
   `renderer_narrow.pt`) and the PersonaPlex weights (Mimi + 7B LM) needed for
   Helium extraction
5. extracts Helium features for the emotion dataset (idempotent)
6. launches emotion LoRA training (Option A: background; Option B: in-notebook),
   with test renders every N epochs using the EMA weights
7. validates the trained adapter (checkpoint contents, emotion bias
   distinctness, null-vs-base alignment) and renders **all 6 emotions + the
   base (emotion-off) reference** with the *live* renderer
8. prints quantitative renderer-compatibility metrics so a renderer that
   cannot visualise the emotion delta is detected numerically

## Requirements

| | |
|---|---|
| **GPU** | NVIDIA GPU with CUDA. ~16 GB VRAM for Helium extraction (bf16 LM; `--quantize_4bit` fits ~8 GB) + ~4 GB for training |
| **Disk** | ~25 GB free (PersonaPlex weights ~16 GB + code + outputs) |
| **HF token** | for `nvidia/personaplex-7b-v1` (gated: accept the licence first) -- skip if the pod already has `weights/mimi.safetensors` + the LM snapshot |
| **Dataset** | `/workspace/emotion_dataset` with `motion/*.pt` [T,32] + **raw audio** (`wav/*.wav`, matching stems) |
| **Test files** | `3robert.jpeg`-style ref not needed (renderer_narrow embeds its base frame); `audio_3.wav` auto-downloaded for the test renders |


In [ ]:
# ============================================================================
# STEP 1 -- Clone the repository with the updated code
# ============================================================================
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL      = "https://github.com/MoshiHead/speech2face_orin_n_original_emotion_adapter_v2.git"
BASE_REPO_URL = "https://github.com/MoshiHead/speech2face_orin_n_original.git"   # base new-pipeline code
REPO_DIR      = "/workspace/speech2face_emotion"


def sh(cmd, cwd=None, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.stdout and r.stdout.strip():
        print(r.stdout.strip()[-3000:])
    if r.stderr and r.stderr.strip():
        print(r.stderr.strip()[-2000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")
    return r


Path(REPO_DIR).parent.mkdir(parents=True, exist_ok=True)

# --- clone (or update) the emotion-adapter repository ------------------------
if Path(REPO_DIR, ".git").exists():
    print(f"[clone] {REPO_DIR} exists -> trying fast-forward update")
    sh("git pull --ff-only", cwd=REPO_DIR, check=False)
else:
    sh(f"git clone {REPO_URL} {REPO_DIR}")

# --- make sure the pipeline code is present ----------------------------------
# The emotion-adapter repo is where the updated code lives. If it does not
# (yet) contain the speech2face sources, bootstrap them from the base repo so
# this notebook is fully self-contained and runnable right now.
STUDENT_MODEL = Path(REPO_DIR, "src", "niloys", "student", "model.py")
LIVE_TRY19 = Path(REPO_DIR, "src", "niloys", "live", "imtalker_try19_student_live.py")
if not STUDENT_MODEL.exists():
    print("[bootstrap] emotion-adapter repo has no pipeline code yet")
    print("[bootstrap] fetching the speech2face base sources ...")
    tmp = tempfile.mkdtemp(prefix="s2f_base_")
    try:
        sh(f"git clone --depth 1 {BASE_REPO_URL} {tmp}")
        src = Path(tmp) / "src"
        dst = Path(REPO_DIR) / "src"
        if src.exists() and not dst.exists():
            shutil.copytree(src, dst)
            print(f"[bootstrap] copied {src} -> {dst}")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

assert STUDENT_MODEL.exists(), f"Student code missing after clone/bootstrap: {STUDENT_MODEL}"
assert LIVE_TRY19.exists(), f"try19 live launcher missing: {LIVE_TRY19}"
print()
print(f"[ready] repository at {REPO_DIR}")
print(f"[ready] student model at {STUDENT_MODEL}")
print(f"[ready] try19 live launcher at {LIVE_TRY19}")


## Configuration

Edit the paths and hyper-parameters below, then run the GPU/dependency cell.
The defaults reproduce the recommended setup (rank-16 LoRA, 10% null anchor,
20k steps).


In [ ]:
# ============================================================================
# CONFIGURATION -- EDIT THE PATHS AND SETTINGS BELOW
# ============================================================================
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

# Cloned repository root (contains src/niloys/, src/imtalker/, src/moshi_pkg/)
REPO_DIR = Path("/workspace/speech2face_emotion")
PROJECT_ROOT = REPO_DIR
SRC_DIR = REPO_DIR / "src"
STUDENT_SRC = SRC_DIR / "niloys" / "student"          # new code is written here
LIVE_SRC = SRC_DIR / "niloys" / "live"
MOSHI_PKG = SRC_DIR / "moshi_pkg"

# Your emotion dataset root: motion/*.pt [T,32] (25 fps) + raw audio files.
# Raw audio is searched in these subfolders (first hit wins):
WAV_DIRS = "wav,audio_wav,raw_audio,wavs,audio_raw"
EMOTION_DATASET_DIR = Path(os.environ.get("EMOTION_DATASET_DIR", "/workspace/emotion_dataset"))
HELIUM_DIR = EMOTION_DATASET_DIR / "helium"           # created by the extraction cell

# Streaming checkpoints (downloaded by the downloads cell if missing)
WEIGHTS_DIR = Path(os.environ.get("S2F_WEIGHTS_DIR", str(PROJECT_ROOT / "weights")))
STUDENT_CKPT = WEIGHTS_DIR / "motion_student_38M.pt"      # frozen base student
RENDERER_CKPT = WEIGHTS_DIR / "renderer_narrow.pt"        # distilled renderer (live)
SILENCE_HELIUM = WEIGHTS_DIR / "silence_helium_mean.pt"   # optional, info only
REF_IMAGE = WEIGHTS_DIR / "ref.jpeg"                      # optional, info only

# PersonaPlex weights for Helium extraction.
# Priority 1: an existing streaming deployment (run_x86.sh layout)
STREAMING_WEIGHTS = Path(os.environ.get("STREAMING_WEIGHTS_DIR", "/workspace/speech2face/weights"))
MIMI_WEIGHT = None     # None = auto-resolve (streaming pod or HF download)
MOSHI_WEIGHT = None    # None = auto-resolve (streaming pod .snap_path or HF download)
PERSONAPLEX_REPO = "nvidia/personaplex-7b-v1"
PERSONAPLEX_REV = "fdaf4090a61cb315c138a1faee287ffd6c716309"
HELIUM_QUANTIZE_4BIT = False   # True = 4-bit LM for extraction (~8 GB VRAM)

# Where to store training outputs (logs, checkpoints, test videos)
EXP_ROOT = PROJECT_ROOT / "emotion_exps"
EXP_NAME = "emotion_student_v1"

# Test assets for the periodic / final renders
TEST_AUDIO = None        # None = auto-resolve audio_3.wav
TEST_HELIUM = None       # None = auto: extracted next to the test audio

# ---------------------------------------------------------------------------
# LoRA hyper-parameters (for emotion conditioning)
# ---------------------------------------------------------------------------
LORA_RANK = 16
LORA_ALPHA = 32.0
LORA_DROPOUT = 0.05
LORA_MLP = True        # LoRA on the block MLPs too (more emotion capacity)
LORA_HEADS = True      # LoRA on head_abs / head_res

# ---------------------------------------------------------------------------
# Training hyper-parameters
# ---------------------------------------------------------------------------
STEPS_PER_CLIP = 100   # Helium steps per training window (100 = 8 s of audio)
BATCH_SIZE = None      # None = auto from GPU VRAM (GPU cell below)
LR = 2e-4
WARMUP = 500
MAX_STEPS = 20_000
MAX_EPOCHS = -1
GRAD_CLIP = 1.0
EMA_DECAY = 0.999
CTX_NOISE = 0.1        # matches the student's own training (ctx_noise=0.1)
NULL_DROP = 0.10       # fraction of samples trained as emotion-off (anchor)
ANCHOR_W = 1.0         # weight of the null-anchor distillation loss
VEL_W = 0.1            # motion velocity (smoothness) loss weight
SAVE_FREQ = 2_500
LOG_EVERY = 50
NUM_WORKERS = 4
VAL_SPLIT = 200
SEED = 0

# ---------------------------------------------------------------------------
# Resume / restart
# ---------------------------------------------------------------------------
AUTO_RESUME = True

# ---------------------------------------------------------------------------
# Emotion filter (optional): None = train on ALL 6 emotions
#   0: neutral, 1: happy, 2: sad, 3: angry, 4: surprise, 5: fear
# ---------------------------------------------------------------------------
EMOTION_FILTER = None

# ---------------------------------------------------------------------------
# CUDA / precision
# ---------------------------------------------------------------------------
TF32 = True            # student trains and runs in fp32 (TF32 matmuls on Ampere+)

# ---------------------------------------------------------------------------
# Periodic test video configuration (every TEST_EVERY_N epochs, EMA weights)
# ---------------------------------------------------------------------------
TEST_EVERY_N = 10
TEST_MOTION_GAIN = 1.2   # same gain the live server applies (GAIN=1.2)

# ---------------------------------------------------------------------------
# Quick sanity prints
# ---------------------------------------------------------------------------
EXP_DIR = EXP_ROOT / EXP_NAME
print(f"repo:                {REPO_DIR}")
print(f"student code:        {STUDENT_SRC}")
print(f"emotion dataset:     {EMOTION_DATASET_DIR}")
print(f"helium features:     {HELIUM_DIR}")
print(f"base student ckpt:   {STUDENT_CKPT}")
print(f"renderer ckpt:       {RENDERER_CKPT}")
print(f"streaming weights:   {STREAMING_WEIGHTS}")
print(f"exp output:          {EXP_DIR}")
print(f"LoRA rank={LORA_RANK} alpha={LORA_ALPHA} mlp={LORA_MLP} heads={LORA_HEADS} | "
      f"batch={BATCH_SIZE} lr={LR} max_steps={MAX_STEPS:,}")
print(f"null_drop={NULL_DROP} anchor_w={ANCHOR_W} ctx_noise={CTX_NOISE} | "
      f"test every {TEST_EVERY_N} epochs")


In [ ]:
# ============================================================================
# GPU DETECTION, CUDA CONFIGURATION, DEPENDENCIES
# ============================================================================
import importlib.util
import os
import shutil
import subprocess
import sys

# --- 1. torch / CUDA ---------------------------------------------------------
try:
    import torch
except ImportError:
    print("[deps] torch not found -> installing CUDA 12.8 build ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0",
                    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU visible to torch. Use a RunPod GPU pod (PyTorch / CUDA image)."
    )

# --- 2. GPU inventory --------------------------------------------------------
GPU_COUNT = torch.cuda.device_count()
GPUS = []
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    GPUS.append({
        "index": i, "name": props.name,
        "vram_gb": props.total_memory / 2**30,
        "cc": (props.major, props.minor),
    })
VRAM_GB = max(g["vram_gb"] for g in GPUS)
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}) | {GPU_COUNT} GPU(s):")
for g in GPUS:
    print(f"  GPU[{g['index']}] {g['name']}  {g['vram_gb']:.1f} GB  "
          f"sm_{g['cc'][0]}{g['cc'][1]}")

# --- 3. VRAM-based batch size -------------------------------------------------
# The student is small; a batch-16 fp32 step (incl. the frozen anchor forward)
# uses ~2 GB. Helium extraction runs in a separate process and frees the LM
# before training starts, so the batch size is NOT limited by the 7B LM.
if BATCH_SIZE is None:
    if VRAM_GB >= 40:   BATCH_SIZE = 32
    elif VRAM_GB >= 24: BATCH_SIZE = 24
    elif VRAM_GB >= 16: BATCH_SIZE = 16
    elif VRAM_GB >= 12: BATCH_SIZE = 12
    else:               BATCH_SIZE = 8
print(f"\nBATCH_SIZE = {BATCH_SIZE}")

# --- 4. CUDA / memory configuration -------------------------------------------
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if TF32:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()
print(f"TF32 = {TF32} | cudnn.benchmark = True | expandable_segments allocator")

# --- 5. Required packages (safe to re-run) -------------------------------------
def _pip(*pkgs):
    print("[install]", " ".join(pkgs))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs), check=True)

required = {
    "numpy":            "numpy",
    "librosa":          "librosa",
    "einops":           "einops",            # moshi_pkg
    "safetensors":      "safetensors",       # moshi loaders
    "sentencepiece":    "sentencepiece",     # moshi stack
    "sphn":             "sphn>=0.2",         # moshi lm import (x86 wheel)
    "huggingface_hub":  "huggingface_hub",
    "cv2":              "opencv-python-headless",
    "PIL":              "Pillow",
    "tqdm":             "tqdm",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    _pip(*missing)
else:
    print("All required packages already installed")

if HELIUM_QUANTIZE_4BIT and importlib.util.find_spec("bitsandbytes") is None:
    _pip("bitsandbytes")   # 4-bit LM for Helium extraction on small GPUs

assert shutil.which("ffmpeg"), "ffmpeg not found on PATH (needed for test videos)"
print("ffmpeg: OK")


## Install the student emotion code into the cloned repo

Six files are written into the cloned repository (all idempotent; re-running
is safe):

1. **`src/niloys/student/emotion_adapter.py`** -- `StudentEmotionConditioner`
   (one-hot(7) -> `d_model` bias; the null index is forced to exact zeros),
   `LoRALinear`, `apply_emotion_lora_to_student()` (freeze all -> LoRA on
   `attn.qkv/proj`, MLPs, `head_abs/head_res`), `install_emotion_on_student()`
   (one patch point covering `forward()` / `step()` / `step_fixed()`),
   `set_emotion()` (in-place runtime switch, CUDA-graph safe) and the
   adapter checkpoint helpers.
2. **`src/niloys/student/emotion_dataset.py`** -- `(helium, prev_motion,
   target, emotion_id)` windows; motion targets are normalised with the
   student's own stats; windows start at a real context frame (offset >= 1)
   to match streaming.
3. **`src/niloys/student/extract_helium.py`** -- batch Helium extraction
   with the live-identical recipe (Mimi -> 8 codebooks -> LM emb sum ->
   transformer, hook at `layers[-2]`).
4. **`src/niloys/student/train_emotion_student.py`** -- the trainer: EMA,
   cosine LR with warmup, grad clip, null-anchor distillation, velocity
   loss, resume, validation, periodic EMA test renders.
5. **`src/niloys/student/test_emotions_student.py`** -- renders base + all
   emotions through the student step loop + renderer_narrow and computes
   renderer-compatibility metrics.
6. **`src/niloys/live/imtalker_try19_student_live_emotion.py`** -- the live
   launcher: try19 + trained emotion adapter + a control-file watcher for
   runtime switching (`echo happy > /workspace/emotion.txt`).

Nothing else in the repo is modified; the patched student remains
backward-compatible (with the null emotion it is the original student by
construction).


In [ ]:
# ============================================================================
# CREATE src/niloys/student/emotion_adapter.py (conditioner + LoRA + install/set_emotion + checkpointing)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""Emotion conditioning adapter for the CausalMotionStudent (Design A).

The CausalMotionStudent has no emotion input, so this module adds one with
the same philosophy that worked for the FMGenerator pipeline:

  * a small trainable conditioner turns the emotion id into a constant
    bias added to the student's input embedding (the `_embed` output --
    the exact analog of the old EmotionConditioner's conditioning-sum bias),
  * targeted LoRA on the attention projections, MLPs and output heads
    gives the emotion pathway capacity,
  * everything else stays frozen so audio/lip-sync behaviour is preserved.

The null emotion (index 6) maps to an exactly-zero bias, so "emotion off"
is the unmodified student *by construction* (the training script adds a
null-anchor distillation loss so the LoRA path also stays aligned with the
frozen base student when no emotion is requested).

All three execution paths of the student funnel through `_embed()`:

  * forward()      -- full-sequence (training)
  * step()         -- live streaming with a KV cache
  * step_fixed()   -- CUDA-graph streaming

so ONE instance-level patch covers training and both live paths.  For the
CUDA-graph path the bias lives in a persistent buffer that `set_emotion()`
mutates in place, so graph replays follow runtime emotion switches.
"""
from __future__ import annotations

import warnings
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# Emotion vocabulary (identical to the old FM pipeline)
# ---------------------------------------------------------------------------
EMOTION_MAP: dict = {
    "neutral":    0,
    "neutrality": 0,
    "happy":      1,
    "happiness":  1,
    "sad":        2,
    "sadness":    2,
    "angry":      3,
    "anger":      3,
    "surprise":   4,
    "surprised":  4,
    "fear":       5,
    "fearful":    5,
}
EMOTION_NAMES: dict = {
    0: "neutral",
    1: "happy",
    2: "sad",
    3: "angry",
    4: "surprise",
    5: "fear",
}
NUM_EMOTIONS = 6          # real classes (0-5)
NULL_EMOTION_IDX = 6      # "no emotion requested" (7-d one-hot, dim_e-1)
DIM_E = NUM_EMOTIONS + 1


def parse_emotion_from_stem(stem: str) -> int:
    """Infer the emotion label from a preprocessed file stem (MEAD naming).

    Splits on '_' and returns the EMOTION_MAP index of the first token that
    matches a known emotion name.  Falls back to 0 (neutral) with a warning.
    """
    for token in stem.lower().split("_"):
        if token in EMOTION_MAP:
            return EMOTION_MAP[token]
    warnings.warn(
        f"[emotion_adapter] could not infer emotion from stem '{stem}'; "
        "defaulting to neutral (0).",
        stacklevel=2,
    )
    return 0


def parse_emotion_value(value) -> int:
    """Accepts an int (0-6), an emotion name ('happy'), or none/null/off."""
    if value is None:
        return NULL_EMOTION_IDX
    if isinstance(value, int) and not isinstance(value, bool):
        v = int(value)
        if 0 <= v <= NULL_EMOTION_IDX:
            return v
        raise ValueError(f"emotion id out of range: {v} (expected 0..{NULL_EMOTION_IDX})")
    s = str(value).strip().lower()
    if s in ("", "none", "null", "off", "-1", "base", "original"):
        return NULL_EMOTION_IDX
    if s.lstrip("-").isdigit():
        return parse_emotion_value(int(s))
    if s in EMOTION_MAP:
        return EMOTION_MAP[s]
    raise ValueError(f"unknown emotion '{value}' -- use 0..5, a name from "
                     f"{sorted(set(EMOTION_NAMES.values()))}, or none/null/off")



# ---------------------------------------------------------------------------
# Conditioner
# ---------------------------------------------------------------------------

class StudentEmotionConditioner(nn.Module):
    """one-hot(dim_e) -> d_model bias, constant over time.

    The final projection is zero-initialised so training starts from the
    exact base-student behaviour.  The null index is forced to exact zeros
    regardless of the learned weights.
    """

    def __init__(self, d_model: int, dim_e: int = DIM_E, dropout: float = 0.0):
        super().__init__()
        self.d_model = int(d_model)
        self.dim_e = int(dim_e)
        self.null_idx = self.dim_e - 1
        self.proj = nn.Sequential(
            nn.Linear(self.dim_e, self.d_model),
            nn.SiLU(),
            nn.Linear(self.d_model, self.d_model),
        )
        nn.init.zeros_(self.proj[-1].weight)
        nn.init.zeros_(self.proj[-1].bias)
        self.drop = nn.Dropout(dropout)

    def forward(self, emotion_id: torch.Tensor, length: int = 1) -> torch.Tensor:
        """emotion_id [B] long -> bias [B, length, d_model]."""
        if emotion_id.ndim == 0:
            emotion_id = emotion_id.unsqueeze(0)
        eid = emotion_id.long().clamp(0, self.dim_e - 1)
        one = F.one_hot(eid, self.dim_e).to(self.proj[0].weight.dtype)
        b = self.proj(self.drop(one))                        # [B, d_model]
        is_null = (eid == self.null_idx).unsqueeze(-1)
        b = torch.where(is_null, torch.zeros_like(b), b)
        return b.unsqueeze(1).expand(-1, int(length), -1).contiguous()


# ---------------------------------------------------------------------------
# LoRA
# ---------------------------------------------------------------------------

class LoRALinear(nn.Module):
    """Low-rank adapter wrapping a frozen nn.Linear (base output + scaled BA)."""

    def __init__(self, base: nn.Linear, rank: int = 16, alpha: float = 32.0,
                 dropout: float = 0.0):
        super().__init__()
        if not isinstance(base, nn.Linear):
            raise TypeError(f"LoRALinear wraps nn.Linear, got {type(base)}")
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank
        self.lora_A = nn.Linear(base.in_features, self.rank, bias=False)
        self.lora_B = nn.Linear(self.rank, base.out_features, bias=False)
        nn.init.normal_(self.lora_A.weight, std=1.0 / self.rank)
        nn.init.zeros_(self.lora_B.weight)                   # identity at init
        self.drop = nn.Dropout(dropout)

    @property
    def in_features(self):
        return self.base.in_features

    @property
    def out_features(self):
        return self.base.out_features

    def forward(self, x):
        return self.base(x) + self.lora_B(self.lora_A(self.drop(x))) * self.scaling



LORA_HEAD_TARGETS = ("head_abs", "head_res")   # head_gate stays frozen (zero-init gate)


def apply_emotion_lora_to_student(model, rank: int = 16, alpha: float = 32.0,
                                  dropout: float = 0.0, include_mlp: bool = True,
                                  include_heads: bool = True) -> int:
    """Freeze the whole student, then LoRA-wrap the targeted Linear layers.

    Targets (per transformer block): attn.qkv, attn.proj, and optionally the
    two MLP Linears; plus head_abs / head_res on the output side.  Audio and
    motion fusion (helium_proj / prev_enc / fuse_*) stay frozen to preserve
    the lip-sync pathway.  Returns the number of wrapped layers.
    """
    for p in model.parameters():
        p.requires_grad_(False)
    wrapped = 0

    def _wrap(owner, attr):
        nonlocal wrapped
        base = getattr(owner, attr)
        if isinstance(base, LoRALinear):
            return  # idempotent
        setattr(owner, attr, LoRALinear(base, rank=rank, alpha=alpha, dropout=dropout))
        wrapped += 1

    for blk in model.blocks:
        _wrap(blk.attn, "qkv")
        _wrap(blk.attn, "proj")
        if include_mlp:
            # mlp = Sequential(Linear, GELU, Dropout, Linear)
            blk.mlp[0] = LoRALinear(blk.mlp[0], rank=rank, alpha=alpha, dropout=dropout)
            blk.mlp[3] = LoRALinear(blk.mlp[3], rank=rank, alpha=alpha, dropout=dropout)
            wrapped += 2
    if include_heads:
        _wrap(model, "head_abs")
        _wrap(model, "head_res")

    model._emotion_lora_cfg = {
        "rank": int(rank), "alpha": float(alpha), "dropout": float(dropout),
        "include_mlp": bool(include_mlp), "include_heads": bool(include_heads),
    }
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)


# ---------------------------------------------------------------------------
# Installation on the student (single patch point for train + both live paths)
# ---------------------------------------------------------------------------

def install_emotion_on_student(model, conditioner: StudentEmotionConditioner | None = None):
    """Attach the conditioner and patch model._embed to add the emotion bias.

    The bias source is model._emotion_bias_state["bias"]:
      * training   -- set per batch to a [B, T, C] tensor (or None),
      * streaming  -- set once to a persistent [1, 1, C] buffer that
                      set_emotion() mutates in place (CUDA-graph safe).

    Idempotent: re-installing replaces the conditioner but patches _embed once.
    """
    if conditioner is None:
        conditioner = StudentEmotionConditioner(model.cfg.d_model)
    model.emotion_conditioner = conditioner      # registered submodule
    if not hasattr(model, "_emotion_bias_state"):
        model._emotion_bias_state = {"bias": None}
    if getattr(model, "_emotion_patched", False):
        return model

    orig_embed = model._embed
    state = model._emotion_bias_state

    def _embed_with_emotion(helium, prev_motion):
        x = orig_embed(helium, prev_motion)
        b = state["bias"]
        if b is not None:
            # slice to the input length: forward() gets [B,T,C], step() gets
            # [B,1,C]; a constant bias is unaffected by the slice.
            x = x + b[:, : x.shape[1]].to(dtype=x.dtype)
        return x

    model._embed = _embed_with_emotion
    model._emotion_patched = True
    return model


def set_emotion(model, value):
    """Runtime emotion switch for streaming (in-place; CUDA-graph safe).

    Accepts 0..5, an emotion name, or none/null/off for the base behaviour.
    The persistent bias buffer is mutated in place so a CUDA graph captured
    over step_fixed() reads the new emotion on the next replay.
    """
    if not getattr(model, "_emotion_patched", False):
        raise RuntimeError("install_emotion_on_student() has not been called")
    eid = parse_emotion_value(value)
    dev = model.helium_mean.device
    buf = getattr(model, "_emotion_stream_buf", None)
    if buf is None:
        buf = torch.zeros(1, 1, model.cfg.d_model, device=dev, dtype=torch.float32)
        model._emotion_stream_buf = buf
        model._emotion_bias_state["bias"] = buf
    with torch.no_grad():
        if eid == NULL_EMOTION_IDX:
            buf.zero_()
        else:
            eid_t = torch.tensor([eid], device=dev, dtype=torch.long)
            b = model.emotion_conditioner(eid_t, length=1)
            buf.copy_(b.to(dtype=buf.dtype))
    model._current_emotion = eid
    return eid


def current_emotion(model) -> int:
    return int(getattr(model, "_current_emotion", NULL_EMOTION_IDX))



# ---------------------------------------------------------------------------
# Checkpointing
# ---------------------------------------------------------------------------

ADAPTER_KEY_HINTS = ("emotion_conditioner", "lora_A", "lora_B")


def adapter_state_dict(model) -> dict:
    """Only the emotion-owned parameters (conditioner + LoRA A/B)."""
    return {k: v.detach().cpu() for k, v in model.state_dict().items()
            if any(h in k for h in ADAPTER_KEY_HINTS)}


def save_emotion_adapter(path, model, cfg_dict: dict, step: int,
                         ema_state: dict | None = None, meta: dict | None = None):
    """Deployment pack: small (adapter-only), loaded on top of the base student."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pack = {
        "adapter": adapter_state_dict(model),
        "ema_adapter": ({k: v.detach().cpu() for k, v in ema_state.items()}
                        if ema_state else None),
        "cfg": dict(cfg_dict),                       # StudentConfig fields
        "lora_cfg": dict(getattr(model, "_emotion_lora_cfg", {})),
        "step": int(step),
        "meta": dict(meta or {}),
    }
    torch.save(pack, str(path))
    return path


def load_emotion_adapter(model, path, prefer_ema: bool = True, verbose: bool = True):
    """Load a trained adapter pack into an emotion-installed student."""
    pack = torch.load(str(path), map_location="cpu", weights_only=False)
    ema = pack.get("ema_adapter")
    state = ema if (prefer_ema and ema) else pack["adapter"]
    missing, _unexpected = model.load_state_dict(state, strict=False)
    missing = [k for k in missing if any(h in k for h in ADAPTER_KEY_HINTS)]
    if missing and verbose:
        print(f"[emotion_adapter] WARNING missing adapter keys: {missing}")
    if verbose:
        which = "ema" if state is ema else "raw"
        print(f"[emotion_adapter] loaded {len(state)} adapter keys ({which}) "
              f"from {path} (step={pack.get('step')})")
    return pack

    print(f"[emotion_lora] wrapped {wrapped} Linear layers "
          f"(rank={rank} alpha={alpha} mlp={include_mlp} heads={include_heads}); "
          f"trainable params so far: {n_train:,}")
    return wrapped
'''

_dst = PROJECT_ROOT / "src/niloys/student/emotion_adapter.py"
_dst.parent.mkdir(parents=True, exist_ok=True)
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# CREATE src/niloys/student/emotion_dataset.py (helium+motion windows for the student)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""Emotion dataset for the CausalMotionStudent (Design A).

Reuses the existing emotion dataset layout from the FM pipeline:

    {root}/motion/{stem}.pt    [T_m, 32]  raw 25 fps motion latents
    {root}/helium/{stem}.npy   [T_h, 4096] raw 12.5 Hz Helium features
                                   (produced by extract_helium.py from the
                                   raw wav files; wav2vec2 audio/*.npy from
                                   the FM dataset is NOT usable here)

The 32-dim motion space is shared by the FMGenerator, the student and the
distilled renderer (the student was distilled from the FMGenerator teacher),
so the motion targets are used directly -- normalised with the student's own
motion_mean / motion_std buffers.

Each sample is a window of `steps` Helium steps (= 2*steps motion frames):

    helium      [steps, 4096]     raw (the student normalises internally)
    prev_motion [steps, 64]       NORMALISED prev-two-frames context
    target      [steps, 2, 32]    NORMALISED target frames
    emotion_id  scalar long (0..5; the training loop applies null dropout)

Windows start at a random step offset >= 1 so the first context frames are
real motion (matching the live streaming state) instead of zeros.
"""
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset

try:
    from student.emotion_adapter import (
        EMOTION_MAP, EMOTION_NAMES, NUM_EMOTIONS, parse_emotion_from_stem,
    )
except ImportError:  # direct execution from the student/ directory
    from emotion_adapter import (
        EMOTION_MAP, EMOTION_NAMES, NUM_EMOTIONS, parse_emotion_from_stem,
    )


class StudentEmotionDataset(Dataset):
    """(helium, prev_motion, target, emotion_id) windows for the student."""

    def __init__(
        self,
        root,
        motion_mean: torch.Tensor,
        motion_std: torch.Tensor,
        steps: int = 100,
        ctx_noise: float = 0.1,
        start: int | None = None,
        end: int | None = None,
        split: str | None = None,
        emotion_filter: list | None = None,
        seed: int = 0,
    ):
        super().__init__()
        self.root = Path(root)
        self.steps = int(steps)
        self.ctx_noise = float(ctx_noise)
        self.motion_mean = motion_mean.float().view(-1)          # [32]
        self.motion_std = motion_std.float().view(-1).clamp_min(1e-5)
        self.rng = random.Random(seed)
        self.split = split or "all"


        motion_dir = self.root / "motion"
        helium_dir = self.root / "helium"
        if not motion_dir.is_dir():
            raise RuntimeError(f"motion dir missing: {motion_dir}")
        if not helium_dir.is_dir():
            raise RuntimeError(
                f"helium dir missing: {helium_dir}\n"
                "Run the Helium extraction cell first (extract_helium.py)."
            )

        motion_files = sorted(motion_dir.glob("*.pt"))
        if start is not None or end is not None:
            motion_files = motion_files[start:end]

        keep_ids = None
        if emotion_filter:
            keep_ids = {EMOTION_MAP[str(n).strip().lower()] for n in emotion_filter}

        self.samples: list = []
        counts = {i: 0 for i in range(NUM_EMOTIONS)}
        skipped = {"no_helium": 0, "too_short": 0, "filtered": 0, "bad": 0}
        for mp in motion_files:
            stem = mp.stem
            hp = helium_dir / f"{stem}.npy"
            if not hp.exists():
                skipped["no_helium"] += 1
                continue
            eid = parse_emotion_from_stem(stem)
            if keep_ids is not None and eid not in keep_ids:
                skipped["filtered"] += 1
                continue
            try:
                t_m = int(torch.load(str(mp), map_location="cpu").shape[0])
                t_h = int(np.load(str(hp), mmap_mode="r").shape[0])
            except Exception as exc:
                print(f"[StudentEmotionDataset] skip {stem}: {exc}")
                skipped["bad"] += 1
                continue
            n_steps = min(t_m // 2, t_h)
            if n_steps < self.steps + 1:
                skipped["too_short"] += 1
                continue
            self.samples.append({
                "stem": stem, "motion_path": str(mp), "helium_path": str(hp),
                "emotion_id": eid, "n_steps": n_steps,
            })
            counts[eid] = counts.get(eid, 0) + 1

        if not self.samples:
            raise RuntimeError(
                f"No usable samples under {self.root} (need motion/*.pt with "
                f"helium/*.npy, each >= {self.steps + 1} Helium steps). "
                f"skipped={skipped}"
            )
        print(f"[StudentEmotionDataset/{self.split}] {len(self.samples)} clips "
              f"(steps={self.steps} ctx_noise={self.ctx_noise}) skipped={skipped}")
        for eid, cnt in sorted(counts.items()):
            if cnt:
                print(f"  {EMOTION_NAMES[eid]:10s} (id={eid}): {cnt}")

    def __len__(self) -> int:
        return len(self.samples)

    def _norm(self, m_raw: torch.Tensor) -> torch.Tensor:
        return (m_raw - self.motion_mean) / self.motion_std

    def __getitem__(self, index: int) -> dict:
        item = self.samples[index]
        motion = torch.load(item["motion_path"], map_location="cpu").float()   # [T_m, 32]
        helium = torch.from_numpy(
            np.load(item["helium_path"], mmap_mode="r").copy()).float()        # [T_h, 4096]

        n_steps = item["n_steps"]
        mn = self._norm(motion[: 2 * n_steps])                                 # [2S, 32]
        helium = helium[:n_steps]

        # random window at offset >= 1 -> first context frames are real motion
        max_o = n_steps - self.steps
        o = self.rng.randint(1, max_o) if max_o > 1 else 1

        h_seg = helium[o: o + self.steps]                                      # [steps, 4096]
        tgt = mn[2 * o: 2 * (o + self.steps)]                                  # [2*steps, 32]
        ctx = mn[2 * o - 2: 2 * (o + self.steps) - 2]                          # [2*steps, 32]

        prev_motion = ctx.view(self.steps, 2, 32).reshape(self.steps, 64).clone()
        target = tgt.view(self.steps, 2, 32).clone()

        if self.ctx_noise > 0:
            prev_motion = prev_motion + torch.randn_like(prev_motion) * self.ctx_noise

        return {
            "helium": h_seg,
            "prev_motion": prev_motion,
            "target": target,
            "emotion_id": torch.tensor(item["emotion_id"], dtype=torch.long),
            "stem": item["stem"],
        }
'''

_dst = PROJECT_ROOT / "src/niloys/student/emotion_dataset.py"
_dst.parent.mkdir(parents=True, exist_ok=True)
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# CREATE src/niloys/student/extract_helium.py (batch Helium extraction, live-identical recipe)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""Batch Helium feature extraction for the emotion dataset.

Produces {dataset}/helium/{stem}.npy [T, 4096] @ 12.5 Hz from raw wav files,
using the EXACT recipe of the live pipeline's HeliumExtractor._extract_raw:

    wav @ 24 kHz -> mimi.encode -> first 8 codebooks
    -> sum_q lm.emb[q](codes[:, q]) + lm.text_emb(padding_ids)
    -> lm.transformer(input_emb) with a forward hook on layers[-2]

This is the same representation the student consumes at live time: the live
direct-reply-hidden path captures the same layer (capture_layer=-2) of the
same PersonaPlex LM, so features extracted here match deployment.

Requires the PersonaPlex weights:
    mimi_weight  = tokenizer-e351c8d8-checkpoint125.safetensors (a.k.a. mimi.safetensors)
    moshi_weight = model.safetensors (the 7B LM, bf16 ~14 GB VRAM;
                   --quantize_4bit brings it under ~6 GB)

Usage:
    python extract_helium.py --dataset_dir /workspace/emotion_dataset \
        --mimi_weight .../mimi.safetensors --moshi_weight .../model.safetensors
    python extract_helium.py --single_wav audio_3.wav --single_out audio_3.npy ...
"""
from __future__ import annotations

import argparse
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch

SAMPLE_RATE = 24000
MAIN_CODEBOOKS = 8


class BatchHeliumExtractor:
    """One-shot (non-streaming) raw Helium extraction; mirrors _extract_raw."""

    def __init__(self, mimi, lm, device):
        self.mimi = mimi
        self.lm = lm
        self.device = device

    @torch.no_grad()
    def extract(self, pcm_np: np.ndarray) -> torch.Tensor:
        wav = torch.from_numpy(np.asarray(pcm_np, dtype=np.float32))
        wav = wav.to(self.device, dtype=torch.float32)[None, None]

        codes = self.mimi.encode(wav)
        codes = codes[:, :MAIN_CODEBOOKS, :].detach()
        batch_size, n_q, total_steps = codes.shape

        dtype = next(self.lm.parameters()).dtype
        input_emb = torch.zeros(batch_size, total_steps, self.lm.dim,
                                device=self.device, dtype=dtype)
        for q in range(n_q):
            input_emb = input_emb + self.lm.emb[q](codes[:, q].long())

        padding_ids = torch.full(
            (batch_size, total_steps),
            self.lm.existing_text_padding_id,
            dtype=torch.long,
            device=self.device,
        )
        input_emb = input_emb + self.lm.text_emb(padding_ids)

        if getattr(self.lm.transformer, "_streaming_state", None) is not None:
            raise RuntimeError("helium lm must stay in batch mode (non-streaming)")

        captured: list = []

        def _hook(_mod, _inp, out):
            captured.append(out.detach())

        handle = self.lm.transformer.layers[-2].register_forward_hook(_hook)
        try:
            self.lm.transformer(input_emb)
        finally:
            handle.remove()

        if len(captured) != 1:
            raise RuntimeError(f"Helium hook captured {len(captured)} tensors; expected 1")
        return captured[0].squeeze(0).float().contiguous()   # [T_raw, 4096]


def load_mimi_and_lm(mimi_weight: str, moshi_weight: str, device,
                     quantize_4bit: bool = False):
    from moshi.models import loaders
    print(f"[helium] loading Mimi from {mimi_weight} ...", flush=True)
    mimi = loaders.get_mimi(str(mimi_weight), device=str(device))
    mimi.eval()
    for p in mimi.parameters():
        p.requires_grad_(False)
    print(f"[helium] loading PersonaPlex LM from {moshi_weight} "
          f"(bf16, quantize_4bit={quantize_4bit}) ...", flush=True)
    lm = loaders.get_moshi_lm(
        str(moshi_weight), device=device, dtype=torch.bfloat16,
        quantize_4bit=quantize_4bit, num_codebooks=MAIN_CODEBOOKS,
    )
    lm.eval()
    for p in lm.parameters():
        p.requires_grad_(False)
    return mimi, lm



def _load_wav(path: Path) -> np.ndarray:
    import librosa
    pcm, _sr = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
    if pcm.size == 0:
        raise RuntimeError(f"empty audio: {path}")
    return np.asarray(pcm, dtype=np.float32)


def _save_npy(feat: torch.Tensor, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_suffix(".npy.tmp")
    np.save(str(tmp), feat.cpu().numpy())
    os.replace(str(tmp), str(out_path))


def find_dataset_wavs(dataset_dir: Path, wav_dir_names: list) -> dict:
    """stem -> wav path, from the first wav dir that contains each stem."""
    found: dict = {}
    for name in wav_dir_names:
        d = dataset_dir / name
        if not d.is_dir():
            continue
        for ext in ("*.wav", "*.mp3", "*.flac", "*.ogg", "*.m4a"):
            for p in sorted(d.glob(ext)):
                found.setdefault(p.stem, p)
    return found


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--dataset_dir", default="",
                   help="emotion dataset root (motion/*.pt stems drive the file list)")
    p.add_argument("--out_dir", default="",
                   help="default: {dataset_dir}/helium")
    p.add_argument("--wav_dirs", default="wav,audio_wav,raw_audio,wavs,audio_raw",
                   help="comma-separated candidate subfolders for raw audio")
    p.add_argument("--single_wav", default="", help="extract just this one file")
    p.add_argument("--single_out", default="", help="output .npy for --single_wav")
    p.add_argument("--mimi_weight", required=True)
    p.add_argument("--moshi_weight", required=True)
    p.add_argument("--quantize_4bit", action="store_true",
                   help="4-bit NF4 trunk (fits ~8 GB GPUs; tiny quality loss)")
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    p.add_argument("--limit", type=int, default=0, help="debug: only N files")
    args = p.parse_args(argv)

    device = torch.device(args.device)

    # ---------------- collect the work list ----------------
    work: list = []   # (wav_path, out_path)
    if args.single_wav:
        if not args.single_out:
            raise SystemExit("--single_out is required with --single_wav")
        work.append((Path(args.single_wav), Path(args.single_out)))
    else:
        if not args.dataset_dir:
            raise SystemExit("give --dataset_dir or --single_wav")
        ds = Path(args.dataset_dir)
        motion_dir = ds / "motion"
        if not motion_dir.is_dir():
            raise SystemExit(f"motion dir missing: {motion_dir}")
        out_dir = Path(args.out_dir) if args.out_dir else ds / "helium"
        wavs = find_dataset_wavs(ds, [s.strip() for s in args.wav_dirs.split(",") if s.strip()])
        stems = [mp.stem for mp in sorted(motion_dir.glob("*.pt"))]
        missing_wav = [s for s in stems if s not in wavs]
        print(f"[helium] dataset: {len(stems)} motion stems, "
              f"{len(wavs)} wav files found, {len(missing_wav)} stems without audio")
        if missing_wav and len(missing_wav) == len(stems):
            raise SystemExit(
                "NO raw audio found for any clip. The student trains on 4096-d\n"
                "Helium features extracted from raw audio -- the dataset's 768-d\n"
                "wav2vec2 audio/*.npy cannot be converted. Place the raw wav\n"
                f"files in one of: {args.wav_dirs} under {ds}\n"
                "(filenames must match the motion/*.pt stems, e.g. M003_happy_level_1_001.wav)."
            )
        for s in stems:
            if s in wavs:
                work.append((wavs[s], out_dir / f"{s}.npy"))

    todo = [(w, o) for (w, o) in work if not o.exists()]
    print(f"[helium] {len(work)} file(s) total, {len(todo)} to extract "
          f"({len(work) - len(todo)} already present)", flush=True)
    if args.limit:
        todo = todo[: args.limit]
    if not todo:
        print("[helium] nothing to do")
        return

    mimi, lm = load_mimi_and_lm(args.mimi_weight, args.moshi_weight, device,
                                quantize_4bit=args.quantize_4bit)
    ex = BatchHeliumExtractor(mimi, lm, device)

    n_done = n_fail = 0
    t_start = time.time()
    for i, (wav_path, out_path) in enumerate(todo):
        try:
            pcm = _load_wav(wav_path)
            feat = ex.extract(pcm)                 # [T, 4096]
            _save_npy(feat, out_path)
            n_done += 1
            if n_done <= 3 or n_done % 50 == 0:
                el = time.time() - t_start
                print(f"[helium] {i + 1}/{len(todo)} {wav_path.stem}: "
                      f"[{feat.shape[0]}, {feat.shape[1]}] "
                      f"({el / max(n_done, 1):.2f}s/file)", flush=True)
        except Exception as exc:
            n_fail += 1
            print(f"[helium] FAIL {wav_path.name}: {exc}", flush=True)

    print(f"[helium] done: extracted={n_done} failed={n_fail} "
          f"in {time.time() - t_start:.1f}s")
    if n_fail:
        raise SystemExit(f"{n_fail} file(s) failed -- see log above")


if __name__ == "__main__":
    main()
'''

_dst = PROJECT_ROOT / "src/niloys/student/extract_helium.py"
_dst.parent.mkdir(parents=True, exist_ok=True)
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# CREATE src/niloys/student/train_emotion_student.py (LoRA training + null anchor + EMA + resume + epoch tests)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""Emotion LoRA training for the CausalMotionStudent (Design A).

Trains ONLY the emotion conditioner + LoRA adapters on top of the frozen
streaming student:

  * emotional samples  -> MSE against the dataset's emotional motion
                          (normalised with the student's own stats)
  * null samples (10%) -> MSE against the FROZEN BASE STUDENT's own output
                          on the same input (null-anchor distillation), so
                          "emotion off" keeps the original live behaviour

The base student is a deterministic 1-step distilled sampler (not flow
matching), so there is no CFG at inference: emotion is a hard condition,
and the null-anchor training is what guarantees the emotion-off path.

Resume: checkpoints/last_train.pt holds step/optimizer/EMA/adapter state.
Deployment artefact: emotion_adapter.pt (adapter-only, EMA weights).
"""
from __future__ import annotations

import argparse
import json
import math
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# allow running as a plain script from the student/ directory
_HERE = Path(__file__).resolve()
for _p in (str(_HERE.parents[1]), str(_HERE.parent)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from student.model import CausalMotionStudent, StudentConfig          # noqa: E402
from student.emotion_adapter import (                                 # noqa: E402
    EMOTION_NAMES, NULL_EMOTION_IDX,
    apply_emotion_lora_to_student, install_emotion_on_student,
    save_emotion_adapter,
)
from student.emotion_dataset import StudentEmotionDataset             # noqa: E402


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--student_ckpt", required=True)
    p.add_argument("--dataset_dir", required=True)
    p.add_argument("--exp_root", required=True)
    p.add_argument("--exp_name", default="emotion_student_v1")
    p.add_argument("--lora_rank", type=int, default=16)
    p.add_argument("--lora_alpha", type=float, default=32.0)
    p.add_argument("--lora_dropout", type=float, default=0.05)
    p.add_argument("--lora_mlp", type=int, default=1)
    p.add_argument("--lora_heads", type=int, default=1)
    p.add_argument("--steps_per_clip", type=int, default=100,
                   help="Helium steps per training window (100 = 8 s)")
    p.add_argument("--ctx_noise", type=float, default=0.1,
                   help="matches the student's own training (ctx_noise=0.1)")
    p.add_argument("--val_split", type=int, default=200)
    p.add_argument("--num_workers", type=int, default=4)
    p.add_argument("--emotion_filter", nargs="*", default=None)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--weight_decay", type=float, default=0.0)
    p.add_argument("--warmup", type=int, default=500)
    p.add_argument("--max_steps", type=int, default=20000)
    p.add_argument("--max_epochs", type=int, default=-1)
    p.add_argument("--grad_clip", type=float, default=1.0)
    p.add_argument("--ema_decay", type=float, default=0.999)
    p.add_argument("--null_drop", type=float, default=0.10,
                   help="fraction of samples trained as emotion-off (anchor)")
    p.add_argument("--anchor_w", type=float, default=1.0)
    p.add_argument("--vel_w", type=float, default=0.1)
    p.add_argument("--save_every", type=int, default=2500)
    p.add_argument("--log_every", type=int, default=50)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--no_auto_resume", action="store_true")
    p.add_argument("--test_every_n_epochs", type=int, default=10)
    p.add_argument("--test_helium", default="", help=".npy [T,4096] for the test audio")
    p.add_argument("--test_audio", default="", help="test wav (muxed into videos)")
    p.add_argument("--renderer_ckpt", default="")
    p.add_argument("--test_motion_gain", type=float, default=1.2)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    return p.parse_args(argv)


def load_frozen_student(ckpt: str, device):
    pack = torch.load(str(ckpt), map_location="cpu", weights_only=False)
    cfg = StudentConfig(**pack["cfg"])
    model = CausalMotionStudent(cfg).to(device).float()
    model.load_state_dict(pack["model"])
    for prm in model.parameters():
        prm.requires_grad_(False)
    model.eval()
    return model, cfg, pack



# ---------------------------------------------------------------------------
# EMA + schedule
# ---------------------------------------------------------------------------

class Ema:
    """Exponential moving average over the trainable parameters."""

    def __init__(self, model, decay: float):
        self.decay = float(decay)
        self.params = {n: p.detach().clone()
                       for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if n in self.params:
                self.params[n].mul_(self.decay).add_(p.detach(), alpha=1.0 - self.decay)

    def state_dict(self):
        return {k: v.cpu() for k, v in self.params.items()}

    def load_state_dict(self, sd):
        for k, v in sd.items():
            if k in self.params:
                self.params[k].copy_(v)

    @torch.no_grad()
    def copy_to(self, model):
        backup = {}
        for n, p in model.named_parameters():
            if n in self.params:
                backup[n] = p.detach().clone()
                p.copy_(self.params[n])
        return backup

    @torch.no_grad()
    def restore(self, model, backup):
        for n, p in model.named_parameters():
            if n in backup:
                p.copy_(backup[n])


def lr_lambda(step: int, warmup: int, max_steps: int):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    t = (step - warmup) / max(1, max_steps - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, t)))


def save_train_checkpoint(path, model, opt, sched, ema, step: int, epoch: int,
                          cfg: StudentConfig, args):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "step": int(step),
        "epoch": int(epoch),
        "optimizer": opt.state_dict(),
        "scheduler": sched.state_dict(),
        "ema": ema.state_dict(),
        "adapter": {k: v.detach().cpu() for k, v in model.state_dict().items()
                    if any(h in k for h in ("emotion_conditioner", "lora_A", "lora_B"))},
        "cfg": {f: getattr(cfg, f) for f in cfg.__dataclass_fields__},
        "lora_cfg": dict(getattr(model, "_emotion_lora_cfg", {})),
        "args": vars(args),
        "rng_torch": torch.get_rng_state(),
        "rng_np": np.random.get_state(),
        "rng_py": random.getstate(),
    }, str(path))



# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------

def main(argv=None):
    args = parse_args(argv)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)
    device = torch.device(args.device)

    exp_dir = Path(args.exp_root) / args.exp_name
    ckpt_dir = exp_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    log_path = exp_dir / "train_log.txt"

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}] {msg}"
        print(line, flush=True)
        with open(log_path, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")

    cfg_dict = None

    def save_deployment(step):
        save_emotion_adapter(
            exp_dir / "emotion_adapter.pt", model, cfg_dict, step,
            ema_state=ema.state_dict(),
            meta={"student_ckpt": Path(args.student_ckpt).name,
                  "null_drop": args.null_drop, "anchor_w": args.anchor_w})

    # ---------------- models ----------------
    log(f"loading base student (anchor): {args.student_ckpt}")
    base, cfg, pack = load_frozen_student(args.student_ckpt, device)
    log(f"  cfg: d_model={cfg.d_model} layers={cfg.n_layers} window={cfg.attn_window} "
        f"lookahead={cfg.lookahead} | teacher step={pack.get('step')}")
    cfg_dict = {f: getattr(cfg, f) for f in cfg.__dataclass_fields__}

    model, _, _ = load_frozen_student(args.student_ckpt, device)
    apply_emotion_lora_to_student(
        model, rank=args.lora_rank, alpha=args.lora_alpha,
        dropout=args.lora_dropout,
        include_mlp=bool(args.lora_mlp), include_heads=bool(args.lora_heads))
    install_emotion_on_student(model)
    model.train()

    trainable = [p for p in model.parameters() if p.requires_grad]
    n_train = sum(p.numel() for p in trainable)
    log(f"trainable params: {n_train:,} / {model.num_params():,} "
        f"({100.0 * n_train / model.num_params():.2f}%)")

    # ---------------- data ----------------
    mm = model.motion_mean.detach().cpu()
    ms = model.motion_std.detach().cpu()
    train_ds = StudentEmotionDataset(
        args.dataset_dir, mm, ms,
        steps=args.steps_per_clip, ctx_noise=args.ctx_noise,
        end=(-args.val_split if args.val_split > 0 else None),
        split="train", emotion_filter=args.emotion_filter, seed=args.seed)
    val_loader = None
    if args.val_split > 0:
        try:
            val_ds = StudentEmotionDataset(
                args.dataset_dir, mm, ms,
                steps=args.steps_per_clip, ctx_noise=0.0,
                start=-args.val_split, split="val",
                emotion_filter=args.emotion_filter, seed=args.seed + 1)
            val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                                    num_workers=0, drop_last=False)
        except RuntimeError as exc:
            log(f"[warn] val split unusable ({exc}); validation disabled")

    def _worker_seed(wid):
        info = torch.utils.data.get_worker_info()
        if info is not None and hasattr(info.dataset, "rng"):
            info.dataset.rng.seed(args.seed + 1000 * wid)

    train_loader = DataLoader(
        train_ds, batch_size=args.batch_size, shuffle=True, drop_last=True,
        num_workers=args.num_workers, pin_memory=True,
        worker_init_fn=_worker_seed, persistent_workers=args.num_workers > 0)
    log(f"steps/epoch = {len(train_loader)} (batch={args.batch_size})")

    # ---------------- optimisation ----------------
    opt = torch.optim.AdamW(trainable, lr=args.lr, betas=(0.9, 0.95),
                            weight_decay=args.weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: lr_lambda(s, args.warmup, args.max_steps))
    ema = Ema(model, args.ema_decay)

    start_step, start_epoch = 0, 0
    last_ck = ckpt_dir / "last_train.pt"
    if last_ck.exists() and not args.no_auto_resume:
        log(f"resuming from {last_ck}")
        ck = torch.load(str(last_ck), map_location="cpu", weights_only=False)
        missing, _unexp = model.load_state_dict(ck["adapter"], strict=False)
        missing = [k for k in missing if "emotion_conditioner" in k or "lora_" in k]
        if missing:
            log(f"[warn] adapter keys missing in resume ckpt: {missing}")
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])
        ema.load_state_dict(ck["ema"])
        start_step = int(ck["step"])
        start_epoch = int(ck.get("epoch", 0))
        try:
            torch.set_rng_state(ck["rng_torch"])
            np.random.set_state(ck["rng_np"])
            random.setstate(ck["rng_py"])
        except Exception as exc:
            log(f"[warn] rng state not restored: {exc}")
        for st in opt.state.values():          # optimizer state -> device
            for k, v in st.items():
                if torch.is_tensor(v):
                    st[k] = v.to(device)
        log(f"resumed at step={start_step} epoch={start_epoch}")

    # ---------------- periodic test render setup ----------------
    renderer = None
    test_helium = None
    if args.test_every_n_epochs > 0:
        if args.test_helium and args.renderer_ckpt:
            from student.test_emotions_student import load_renderer, render_test_set
            renderer = load_renderer(args.renderer_ckpt, device)
            test_helium = torch.from_numpy(np.load(args.test_helium)).float()
            log(f"test renders every {args.test_every_n_epochs} epochs "
                f"(helium {tuple(test_helium.shape)})")
        else:
            log("[warn] test renders requested but --test_helium/--renderer_ckpt "
                "missing; disabled")

    import signal as _signal
    _stop = {"flag": False}

    def _sigterm(_sig, _frm):
        _stop["flag"] = True
        log("SIGTERM received -- finishing the current step, saving, exiting")

    try:
        _signal.signal(_signal.SIGTERM, _sigterm)
    except (ValueError, OSError):
        pass   # not in main thread / unsupported platform

    # ---------------- training loop ----------------
    log(f"training: max_steps={args.max_steps} lr={args.lr} warmup={args.warmup} "
        f"null_drop={args.null_drop} anchor_w={args.anchor_w} vel_w={args.vel_w} "
        f"ema={args.ema_decay}")
    step = start_step
    epoch = start_epoch
    t_train = time.time()
    done = False

    while step < args.max_steps and not done:
        epoch += 1
        if 0 < args.max_epochs < epoch:
            break
        model.train()
        for batch in train_loader:
            if step >= args.max_steps or _stop["flag"]:
                done = True
                break
            helium = batch["helium"].to(device, non_blocking=True)
            prev = batch["prev_motion"].to(device, non_blocking=True)
            target = batch["target"].to(device, non_blocking=True)
            eid = batch["emotion_id"].to(device, non_blocking=True)
            B, T = helium.shape[0], helium.shape[1]

            null_mask = torch.rand(B, device=device) < args.null_drop
            eid_eff = torch.where(
                null_mask, torch.full_like(eid, NULL_EMOTION_IDX), eid)

            bias = model.emotion_conditioner(eid_eff, length=T)
            model._emotion_bias_state["bias"] = bias
            out, _a, _r, _g = model(helium, prev)
            model._emotion_bias_state["bias"] = None

            emo_mask = ~null_mask
            loss = torch.zeros((), device=device)
            parts = {}
            if emo_mask.any():
                loss_e = F.mse_loss(out[emo_mask], target[emo_mask])
                loss = loss + loss_e
                parts["loss_e"] = loss_e.item()
                if args.vel_w > 0:
                    of_ = out[emo_mask].reshape(-1, T * 2, cfg.motion_dim)
                    tg_ = target[emo_mask].reshape(-1, T * 2, cfg.motion_dim)
                    loss_v = F.mse_loss(of_[:, 1:] - of_[:, :-1],
                                        tg_[:, 1:] - tg_[:, :-1])
                    loss = loss + args.vel_w * loss_v
                    parts["loss_v"] = loss_v.item()
            if null_mask.any():
                with torch.no_grad():
                    base_out, _, _, _ = base(helium[null_mask], prev[null_mask])
                loss_n = F.mse_loss(out[null_mask], base_out)
                loss = loss + args.anchor_w * loss_n
                parts["loss_n"] = loss_n.item()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(trainable, args.grad_clip)
            opt.step()
            sched.step()
            ema.update(model)
            step += 1

            if step % args.log_every == 0:
                lr_now = sched.get_last_lr()[0]
                msg = (f"step={step}/{args.max_steps} epoch={epoch} "
                       + " ".join(f"{k}={v:.5f}" for k, v in parts.items())
                       + f" gn={float(gn):.3f} lr={lr_now:.2e}")
                log(msg)
            if step % args.save_every == 0:
                save_train_checkpoint(ckpt_dir / f"step-{step:07d}.pt",
                                      model, opt, sched, ema, step, epoch, cfg, args)
                save_train_checkpoint(last_ck, model, opt, sched, ema, step, epoch,
                                      cfg, args)
                save_deployment(step)
                log(f"saved checkpoints (step={step})")

        # ---------------- end-of-epoch: validation ----------------
        if val_loader is not None:
            model.eval()
            ve_sum = vn_sum = nv = 0
            with torch.no_grad():
                for vb_i, vb in enumerate(val_loader):
                    if vb_i >= 20:
                        break
                    vh = vb["helium"].to(device)
                    vp = vb["prev_motion"].to(device)
                    vt = vb["target"].to(device)
                    ve = vb["emotion_id"].to(device)
                    vB, vT = vh.shape[0], vh.shape[1]
                    vbias = model.emotion_conditioner(ve, length=vT)
                    model._emotion_bias_state["bias"] = vbias
                    vout = model(vh, vp)[0]
                    ve_sum += F.mse_loss(vout, vt).item() * vB
                    model._emotion_bias_state["bias"] = None
                    vbase = base(vh, vp)[0]
                    vbias_null = model.emotion_conditioner(
                        torch.full_like(ve, NULL_EMOTION_IDX), length=vT)
                    model._emotion_bias_state["bias"] = vbias_null
                    vnull = model(vh, vp)[0]
                    model._emotion_bias_state["bias"] = None
                    vn_sum += F.mse_loss(vnull, vbase).item() * vB
                    nv += vB
            model.train()
            log(f"VAL epoch={epoch} val_loss_e={ve_sum / max(nv, 1):.5f} "
                f"val_null_vs_base={vn_sum / max(nv, 1):.5f} (n={nv})")

        # ---------------- end-of-epoch: test renders (EMA weights) ---------
        if (renderer is not None and test_helium is not None
                and epoch % args.test_every_n_epochs == 0):
            log(f"rendering epoch-{epoch} test videos (EMA weights) ...")
            backup = ema.copy_to(model)
            model.eval()
            try:
                render_test_set(
                    model, cfg, renderer, test_helium,
                    exp_dir / "test_videos" / f"epoch_{epoch:04d}",
                    audio_path=args.test_audio or None,
                    emotions=(None, 1, 2, 3),
                    motion_gain=args.test_motion_gain,
                    tag=f"e{epoch:04d}")
            except Exception as exc:
                log(f"[warn] epoch test render failed: {exc}")
            ema.restore(model, backup)
            model.train()

    # ---------------- final save ----------------
    save_train_checkpoint(last_ck, model, opt, sched, ema, step, epoch, cfg, args)
    save_deployment(step)
    log(f"DONE step={step} in {(time.time() - t_train) / 60.0:.1f} min")
    log(f"deployment pack: {exp_dir / 'emotion_adapter.pt'}")


if __name__ == "__main__":
    main()
'''

_dst = PROJECT_ROOT / "src/niloys/student/train_emotion_student.py"
_dst.parent.mkdir(parents=True, exist_ok=True)
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# CREATE src/niloys/student/test_emotions_student.py (render all emotions + renderer-compatibility metrics)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""Render emotion test videos for the CausalMotionStudent + distilled renderer.

Pipeline (identical to the live try19 server):

    helium [T, 4096] @ 12.5 Hz
      -> emotion student step loop (init_state + step per Helium step)
      -> denorm_motion -> raw [2T, 32] @ 25 fps
      -> renderer_narrow (embedded base_frame + stats, blink phase)
      -> 512x512 frames -> ffmpeg mp4 with the source audio

It also reports quantitative renderer-compatibility metrics so a renderer
that cannot visualise the emotion delta is detected numerically instead of
assumed:

    motion_l2_vs_base  -- mean per-frame L2 of the raw motion vs the base
                          (emotion-off) student output
    pixel_mad_vs_base  -- mean |pixel| difference (0..255) of the rendered
                          frames vs the base render

If motion_l2 is healthy but pixel_mad is tiny, the distilled renderer is
not translating the emotion motion into visible pixels and needs
fine-tuning (see the notebook's validation section).
"""
from __future__ import annotations

import argparse
import json
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import torch

# allow running as a plain script from the student/ directory
_HERE = Path(__file__).resolve()
for _p in (str(_HERE.parents[1]), str(_HERE.parent)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from student.model import CausalMotionStudent, StudentConfig          # noqa: E402
from student.renderer import build as build_renderer                  # noqa: E402
from student.emotion_adapter import (                                 # noqa: E402
    EMOTION_NAMES, NULL_EMOTION_IDX,
    apply_emotion_lora_to_student, install_emotion_on_student,
    load_emotion_adapter, set_emotion,
)


def load_emotion_student(student_ckpt: str, emotion_ckpt: str | None, device,
                         verbose: bool = True):
    """Base student (+ optional trained emotion adapter), eval mode, frozen."""
    pack = torch.load(str(student_ckpt), map_location="cpu", weights_only=False)
    cfg = StudentConfig(**pack["cfg"])
    model = CausalMotionStudent(cfg).to(device).float().eval()
    model.load_state_dict(pack["model"])
    if emotion_ckpt:
        lpack = torch.load(str(emotion_ckpt), map_location="cpu", weights_only=False)
        lc = lpack.get("lora_cfg") or {}
        apply_emotion_lora_to_student(
            model,
            rank=int(lc.get("rank", 16)), alpha=float(lc.get("alpha", 32.0)),
            dropout=0.0, include_mlp=bool(lc.get("include_mlp", True)),
            include_heads=bool(lc.get("include_heads", True)),
        )
        install_emotion_on_student(model)
        load_emotion_adapter(model, emotion_ckpt, prefer_ema=True, verbose=verbose)
    else:
        install_emotion_on_student(model)
    for p in model.parameters():
        p.requires_grad_(False)
    if verbose:
        print(f"[test] student: {student_ckpt} (params={model.num_params():,})")
    return model, cfg


def load_renderer(renderer_ckpt: str, device, verbose: bool = True):
    pack = torch.load(str(renderer_ckpt), map_location="cpu", weights_only=False)
    r = build_renderer(**pack["cfg"]).to(device).float().eval()
    r.load_state_dict(pack["model"])
    for p in r.parameters():
        p.requires_grad_(False)
    if verbose:
        print(f"[test] renderer: {renderer_ckpt} (val_psnr={pack.get('val_psnr')})")
    return r


@torch.no_grad()
def sample_motion(model, cfg, helium: torch.Tensor, emotion_id: int,
                  motion_gain: float = 1.0) -> torch.Tensor:
    """helium [T,4096] -> RAW motion [2T, 32] (same step loop as live)."""
    device = model.helium_mean.device
    set_emotion(model, emotion_id)
    h = helium.to(device=device, dtype=torch.float32).unsqueeze(0)   # [1, T, 4096]
    st = model.init_state(1, device, torch.float32)
    pairs = [model.step(h[:, t], st) for t in range(int(h.shape[1]))]
    pred_n = torch.stack(pairs, 1)                              # [1, T, 2, 32]
    if motion_gain != 1.0:
        mu = pred_n.mean(dim=1, keepdim=True)
        pred_n = mu + (pred_n - mu) * float(motion_gain)
    motion = model.denorm_motion(pred_n)[0].reshape(-1, 32)     # [2T, 32] raw
    set_emotion(model, NULL_EMOTION_IDX)                        # leave clean
    return motion.detach()


@torch.no_grad()
def render_frames(renderer, motion: torch.Tensor, batch: int = 50) -> np.ndarray:
    """RAW motion [T,32] -> uint8 frames [T,512,512,3] (blink phase like live)."""
    device = motion.device
    mo = motion.to(device=device, dtype=torch.float32)
    bl = int(getattr(renderer.cfg, "blink_len", 0) or 0)
    outs = []
    for i in range(0, int(mo.shape[0]), batch):
        chunk = mo[i: i + batch]
        if bl > 0:
            ph = (torch.arange(i, i + chunk.shape[0], device=device)) % bl
            img = renderer(chunk, ph)
        else:
            img = renderer(chunk)
        outs.append(img.clamp(0, 1).float().cpu())
    frames = torch.cat(outs, 0)
    return frames.mul(255).to(torch.uint8).permute(0, 2, 3, 1).numpy()



def write_video(frames: np.ndarray, audio_path: str | None, out_path: Path,
                fps: int = 25):
    """uint8 frames [T,H,W,3] + optional audio -> mp4 (cv2 + ffmpeg mux)."""
    import cv2
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix="emo_vid_"))
    silent = tmp_dir / "silent.mp4"
    h, w = frames.shape[1:3]
    vw = cv2.VideoWriter(str(silent), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for f in frames:
        vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    vw.release()
    if audio_path:
        subprocess.run(
            ["ffmpeg", "-y", "-v", "warning",
             "-i", str(silent), "-i", str(audio_path),
             "-c:v", "copy", "-c:a", "aac", "-shortest", str(out_path)],
            check=True,
        )
    else:
        subprocess.run(
            ["ffmpeg", "-y", "-v", "warning", "-i", str(silent),
             "-c:v", "copy", str(out_path)],
            check=True,
        )
    try:
        silent.unlink()
        tmp_dir.rmdir()
    except OSError:
        pass
    return out_path


def render_test_set(student, cfg, renderer, helium: torch.Tensor,
                    out_dir: Path, audio_path: str | None = None,
                    emotions=(None, 1, 2, 3), motion_gain: float = 1.0,
                    tag: str = "") -> dict:
    """Render base (emotion off) + the requested emotions; return metrics.

    emotions: iterable of None (= base / null) or emotion ids 0..5.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    results: dict = {}
    base_motion = None
    base_frames = None

    for e in emotions:
        eid = NULL_EMOTION_IDX if e is None else int(e)
        name = "base" if e is None else f"{eid}_{EMOTION_NAMES[eid]}"
        t0 = time.time()
        motion = sample_motion(student, cfg, helium, eid, motion_gain=motion_gain)
        frames = render_frames(renderer, motion)
        if e is None:
            base_motion, base_frames = motion, frames
        fname = f"{tag + '_' if tag else ''}{name}.mp4"
        write_video(frames, audio_path, out_dir / fname)
        dt = time.time() - t0
        info = {"video": str(out_dir / fname), "frames": int(frames.shape[0])}
        if base_motion is not None and e is not None:
            info["motion_l2_vs_base"] = float(
                (motion - base_motion).norm(dim=-1).mean().item())
            info["pixel_mad_vs_base"] = float(
                np.abs(frames.astype(np.float32) - base_frames.astype(np.float32)).mean())
        print(f"[test] {name}: {frames.shape[0]} frames in {dt:.1f}s -> {fname}  "
              + (f"motion_l2={info.get('motion_l2_vs_base', 0):.4f} "
                 f"pixel_mad={info.get('pixel_mad_vs_base', 0):.3f}" if e is not None
                 else "(reference)"))
        results[name] = info
    return results


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--student_ckpt", required=True)
    p.add_argument("--emotion_ckpt", default="",
                   help="trained emotion adapter pack (empty = untrained sanity run)")
    p.add_argument("--renderer_ckpt", required=True)
    p.add_argument("--helium", required=True,
                   help=".npy [T, 4096] Helium features for the test audio")
    p.add_argument("--audio", default="", help="source audio (muxed into the mp4)")
    p.add_argument("--out_dir", required=True)
    p.add_argument("--motion_gain", type=float, default=1.2,
                   help="same gain the live server applies (GAIN=1.2)")
    p.add_argument("--emotions", default="all",
                   help="'all' or comma-separated ids, e.g. 1,3,4")
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    p.add_argument("--seed", type=int, default=0)
    args = p.parse_args(argv)

    torch.manual_seed(args.seed)
    device = torch.device(args.device)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    student, cfg = load_emotion_student(
        args.student_ckpt, args.emotion_ckpt or None, device)
    renderer = load_renderer(args.renderer_ckpt, device)

    helium = torch.from_numpy(np.load(args.helium)).float()     # [T, 4096]
    assert helium.ndim == 2 and helium.shape[1] == 4096, helium.shape
    print(f"[test] helium: {tuple(helium.shape)} ({helium.shape[0] / 12.5:.1f}s)")

    if args.emotions.strip().lower() == "all":
        emo_list = [None, 0, 1, 2, 3, 4, 5]
    else:
        emo_list = [None] + [int(x) for x in args.emotions.split(",") if x.strip()]

    results = render_test_set(
        student, cfg, renderer, helium, out_dir,
        audio_path=args.audio or None, emotions=emo_list,
        motion_gain=args.motion_gain,
    )

    # ---- renderer compatibility verdict ----------------------------------
    metrics_path = out_dir / "metrics.json"
    metrics_path.write_text(json.dumps(results, indent=2))
    emo = {k: v for k, v in results.items() if k != "base" and "pixel_mad_vs_base" in v}
    print("\n=== renderer compatibility ===")
    if not emo:
        print("no emotional renders -- nothing to compare")
        return
    ml2 = np.mean([v["motion_l2_vs_base"] for v in emo.values()])
    pmad = np.mean([v["pixel_mad_vs_base"] for v in emo.values()])
    print(f"mean motion_l2_vs_base = {ml2:.4f}  (motion space delta vs emotion-off)")
    print(f"mean pixel_mad_vs_base = {pmad:.3f} (0..255 pixel scale)")
    if pmad < 0.5:
        print("[WARN] the rendered pixel delta is very small -- the distilled\n"
              "       renderer may not be visualising the emotion motion.\n"
              "       If the videos look identical, the renderer needs\n"
              "       fine-tuning on emotional motion (see the notebook).")
    else:
        print("[OK] emotion delta is visible in the rendered pixels.")
    print(f"metrics written to {metrics_path}")


if __name__ == "__main__":
    main()
'''

_dst = PROJECT_ROOT / "src/niloys/student/test_emotions_student.py"
_dst.parent.mkdir(parents=True, exist_ok=True)
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# CREATE src/niloys/live/imtalker_try19_student_live_emotion.py (live launcher with runtime emotion switching)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""Emotion-enabled twin of imtalker_try19_student_live.py.

Runs the exact same streaming stack (CausalMotionStudent + renderer_narrow
on the niloys live server) with a trained emotion adapter installed in the
student, and adds RUNTIME EMOTION SWITCHING through a control file:

    echo happy    > /workspace/emotion.txt     # switch to happy
    echo 3        > /workspace/emotion.txt     # switch to angry
    echo none     > /workspace/emotion.txt     # back to the original behaviour

No existing file is modified: this module wraps try19's load_student() and
then calls try19.main() unchanged.  All try19/server arguments work exactly
as before; the three extra arguments below are stripped from sys.argv first.

Extra arguments (or environment variables):
    --emotion_ckpt PATH   trained emotion adapter pack (emotion_adapter.pt)
                          [env EMOTION_CKPT] -- REQUIRED
    --emotion_file PATH   control file to watch   [env EMOTION_FILE]
                          default /workspace/emotion.txt
    --emotion VALUE       initial emotion: 0..5, a name, or none
                          [env EMOTION_INIT]      default none

The switch is CUDA-graph safe: the emotion bias lives in a persistent
buffer that is mutated in place, so graph replays over step_fixed() read
the new emotion on the next step (the KV history keeps the previous
emotion's context, which gives a natural transition over ~1-2 s).
"""
from __future__ import annotations

import os
import sys
import threading
import time
from pathlib import Path

import torch

_THIS = Path(__file__).resolve()
_SRC_NILOYS = _THIS.parents[1]          # src/niloys
_LIVE_DIR = _THIS.parent                # src/niloys/live
for _p in (str(_SRC_NILOYS), str(_LIVE_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ---------------------------------------------------------------------------
# strip our extra args from sys.argv BEFORE try19/server argparse sees them
# ---------------------------------------------------------------------------
_OWN_ARGS = ("--emotion_ckpt", "--emotion_file", "--emotion")


def _strip_own_args(argv):
    own, rest = {}, []
    i = 0
    while i < len(argv):
        a = argv[i]
        if a in _OWN_ARGS and i + 1 < len(argv):
            own[a] = argv[i + 1]
            i += 2
        elif any(a.startswith(k + "=") for k in _OWN_ARGS):
            k, v = a.split("=", 1)
            own[k] = v
            i += 1
        else:
            rest.append(a)
            i += 1
    return own, rest


_own, _rest = _strip_own_args(sys.argv[1:])
EMOTION_CKPT = _own.get("--emotion_ckpt") or os.environ.get("EMOTION_CKPT", "")
EMOTION_FILE = (_own.get("--emotion_file") or
                os.environ.get("EMOTION_FILE", "/workspace/emotion.txt"))
EMOTION_INIT = _own.get("--emotion") or os.environ.get("EMOTION_INIT", "none")
sys.argv = [sys.argv[0]] + _rest

if not EMOTION_CKPT:
    raise SystemExit(
        "[emotion] --emotion_ckpt (or EMOTION_CKPT) is required: path to the\n"
        "trained emotion_adapter.pt produced by train_emotion_student.py"
    )

import imtalker_try19_student_live as try19                          # noqa: E402
from student.emotion_adapter import (                                # noqa: E402
    EMOTION_NAMES, NULL_EMOTION_IDX,
    apply_emotion_lora_to_student, install_emotion_on_student,
    load_emotion_adapter, parse_emotion_value, set_emotion,
)


# ---------------------------------------------------------------------------
# wrap try19.load_student: install the trained emotion adapter on the student
# ---------------------------------------------------------------------------
_holder = {"model": None}
_orig_load_student = try19.load_student


def load_student_with_emotion(ckpt, device):
    model, cfg = _orig_load_student(ckpt, device)
    pack = torch.load(str(EMOTION_CKPT), map_location="cpu", weights_only=False)
    lc = pack.get("lora_cfg") or {}
    apply_emotion_lora_to_student(
        model,
        rank=int(lc.get("rank", 16)),
        alpha=float(lc.get("alpha", 32.0)),
        dropout=0.0,
        include_mlp=bool(lc.get("include_mlp", True)),
        include_heads=bool(lc.get("include_heads", True)),
    )
    install_emotion_on_student(model)
    load_emotion_adapter(model, EMOTION_CKPT, prefer_ema=True)
    for prm in model.parameters():
        prm.requires_grad_(False)
    eid = set_emotion(model, EMOTION_INIT)   # allocates the persistent buffer
    _holder["model"] = model
    name = "none (base behaviour)" if eid == NULL_EMOTION_IDX else EMOTION_NAMES[eid]
    print(f"[emotion] adapter installed from {EMOTION_CKPT}", flush=True)
    print(f"[emotion] initial emotion: {EMOTION_INIT} -> {name}", flush=True)
    return model, cfg


try19.load_student = load_student_with_emotion


# ---------------------------------------------------------------------------
# runtime switching: watch the control file
# ---------------------------------------------------------------------------

def _watch_control_file():
    path = Path(EMOTION_FILE)
    last_raw = None
    while True:
        raw = None
        try:
            if path.exists():
                raw = path.read_text(encoding="utf-8", errors="replace").strip()
                if raw and raw != last_raw:
                    model = _holder["model"]
                    if model is not None:
                        eid = set_emotion(model, raw)
                        name = ("none (base behaviour)" if eid == NULL_EMOTION_IDX
                                else EMOTION_NAMES[eid])
                        print(f"[emotion] switched -> '{raw}' = {name}", flush=True)
                        last_raw = raw
        except Exception as exc:
            print(f"[emotion] control-file error ({exc}); "
                  f"use 0..5, one of {sorted(set(EMOTION_NAMES.values()))}, or none",
                  flush=True)
            if raw is not None:
                last_raw = raw   # don't spam the same invalid value
        time.sleep(0.5)


threading.Thread(target=_watch_control_file, daemon=True).start()

print("=" * 72, flush=True)
print("[emotion] RUNTIME EMOTION SWITCHING ENABLED", flush=True)
print(f"[emotion]   control file: {EMOTION_FILE}", flush=True)
print("[emotion]   echo happy    > {0}   # 0 neutral 1 happy 2 sad".format(EMOTION_FILE), flush=True)
print("[emotion]   echo 3        > {0}   # 3 angry 4 surprise 5 fear".format(EMOTION_FILE), flush=True)
print("[emotion]   echo none     > {0}   # original behaviour".format(EMOTION_FILE), flush=True)
print("=" * 72, flush=True)

try19.main()

'''

_dst = PROJECT_ROOT / "src/niloys/live/imtalker_try19_student_live_emotion.py"
_dst.parent.mkdir(parents=True, exist_ok=True)
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# Verify the student emotion code is importable and wires into the student
# ============================================================================
import inspect
import sys
from pathlib import Path

for _p in (str(SRC_DIR / "niloys"), str(SRC_DIR / "niloys" / "student"),
           str(SRC_DIR / "niloys" / "live"), str(MOSHI_PKG)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from student.model import CausalMotionStudent, StudentConfig
print("[OK] student/model.py: CausalMotionStudent")

from student.emotion_adapter import (
    EMOTION_MAP, EMOTION_NAMES, NUM_EMOTIONS, NULL_EMOTION_IDX,
    LoRALinear, StudentEmotionConditioner,
    apply_emotion_lora_to_student, install_emotion_on_student,
    load_emotion_adapter, parse_emotion_from_stem, parse_emotion_value,
    save_emotion_adapter, set_emotion,
)
print("[OK] student/emotion_adapter.py: conditioner + LoRA + set_emotion")

from student.emotion_dataset import StudentEmotionDataset
print("[OK] student/emotion_dataset.py: StudentEmotionDataset")

import student.extract_helium as _ex
assert hasattr(_ex, "BatchHeliumExtractor") and hasattr(_ex, "load_mimi_and_lm")
print("[OK] student/extract_helium.py: BatchHeliumExtractor + load_mimi_and_lm")

import student.train_emotion_student as _tr
assert hasattr(_tr, "main") and hasattr(_tr, "parse_args")
print("[OK] student/train_emotion_student.py: main()/parse_args()")

import student.test_emotions_student as _ts
for _f in ("load_emotion_student", "load_renderer", "sample_motion",
           "render_frames", "render_test_set", "main"):
    assert hasattr(_ts, _f), _f
print("[OK] student/test_emotions_student.py: render + metrics helpers")

_spec = LIVE_SRC / "imtalker_try19_student_live_emotion.py"
compile(_spec.read_text(encoding="utf-8"), str(_spec), "exec")
print("[OK] live/imtalker_try19_student_live_emotion.py: compiles")

# --- functional check on a fresh (untrained) student -------------------------
import torch

cfg = StudentConfig()
model = CausalMotionStudent(cfg)
apply_emotion_lora_to_student(model, rank=LORA_RANK, alpha=LORA_ALPHA,
                              dropout=LORA_DROPOUT, include_mlp=LORA_MLP,
                              include_heads=LORA_HEADS)
install_emotion_on_student(model)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[OK] emotion install: {n_train:,} trainable params "
      f"({100.0 * n_train / model.num_params():.2f}% of {model.num_params():,})")

# null emotion must be an exact no-op at init (zero-init conditioner + LoRA)
x_h = torch.randn(1, 4, cfg.helium_dim)
x_pm = torch.randn(1, 4, cfg.out_dim)
model.eval()
with torch.no_grad():
    out_none = model(x_h, x_pm)[0]
    model._emotion_bias_state["bias"] = model.emotion_conditioner(
        torch.tensor([NULL_EMOTION_IDX]), length=4)
    out_null = model(x_h, x_pm)[0]
    model._emotion_bias_state["bias"] = None
assert torch.equal(out_none, out_null), "null emotion is not an exact no-op"
print("[OK] null emotion is an EXACT no-op at init")

# set_emotion allocates the persistent streaming buffer
set_emotion(model, "happy")
assert model._emotion_stream_buf is not None
assert set_emotion(model, "none") == NULL_EMOTION_IDX
assert float(model._emotion_stream_buf.abs().max()) == 0.0
print("[OK] set_emotion: names/ids/none + persistent buffer")

print("\nAll student emotion code verified.")


## Download the streaming checkpoints and PersonaPlex weights

Downloads / resolves everything needed for training and Helium extraction:

| File | Source | Used for |
|---|---|---|
| `motion_student_38M.pt` | `niloy629/personaplex-parallel-depformer` (public) | the frozen base student (+ null anchor) |
| `renderer_narrow.pt` | same | test renders (the live renderer) |
| `mimi.safetensors` | existing streaming pod, or `nvidia/personaplex-7b-v1` (gated) | Helium extraction |
| `model.safetensors` (7B LM) | existing pod `.snap_path`, or the same gated repo | Helium extraction |
| `audio_3.wav` | next to this notebook, or the old pipeline repo | test renders |

If this pod already runs the streaming server (`/workspace/speech2face` with
`weights/` + `.snap_path`), the PersonaPlex weights are **reused** and no HF
token is needed. Otherwise set `HF_TOKEN` (the repo is gated: accept the
licence on the model page first).


In [ ]:
# ============================================================================
# Download / resolve all checkpoints (autonomous, idempotent)
# ============================================================================
import os
import shutil
import urllib.request
from pathlib import Path
from huggingface_hub import hf_hub_download

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

AVATAR_REPO = "niloy629/personaplex-parallel-depformer"

# --- 1. streaming avatar checkpoints (public) --------------------------------
for folder, fname in [("avatar_assets", "motion_student_38M.pt"),
                      ("avatar_assets", "renderer_narrow.pt"),
                      ("avatar_assets", "ref.jpeg"),
                      ("avatar_assets", "silence_helium_mean.pt")]:
    dest = WEIGHTS_DIR / fname
    if not dest.exists():
        print(f"[download] {fname} from {AVATAR_REPO} ...")
        src = hf_hub_download(repo_id=AVATAR_REPO, filename=f"{folder}/{fname}")
        shutil.copy2(src, dest)
        print(f"  [OK] -> {dest}")
    else:
        print(f"[skip] {fname} already present")

# --- 2. PersonaPlex weights (Mimi + 7B LM) ------------------------------------
def _resolve_personaplex():
    """Return (mimi_path, lm_path) -- prefer an existing streaming deployment."""
    mimi_cand = []
    lm_cand = []
    if MIMI_WEIGHT:
        mimi_cand.append(Path(MIMI_WEIGHT))
    if MOSHI_WEIGHT:
        lm_cand.append(Path(MOSHI_WEIGHT))
    mimi_cand += [STREAMING_WEIGHTS / "mimi.safetensors"]
    snap_file = STREAMING_WEIGHTS.parent / ".snap_path"
    if snap_file.exists():
        snap = Path(snap_file.read_text().strip())
        lm_cand.append(snap / "model.safetensors")
    mimi = next((p for p in mimi_cand if p.exists()), None)
    lm = next((p for p in lm_cand if p.exists()), None)
    return mimi, lm

MIMI_WEIGHT_R, MOSHI_WEIGHT_R = _resolve_personaplex()
if MIMI_WEIGHT_R and MOSHI_WEIGHT_R:
    print(f"[reuse] Mimi: {MIMI_WEIGHT_R}")
    print(f"[reuse] LM:   {MOSHI_WEIGHT_R}")
else:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not token:
        try:
            from huggingface_hub import get_token
            token = get_token()
        except Exception:
            token = None
    assert token, (
        "PersonaPlex weights not found locally and no HF token available.\n"
        "Set HF_TOKEN (the nvidia/personaplex-7b-v1 repo is gated -- accept the\n"
        "licence on the model page first), or run on a pod that already has the\n"
        "streaming deployment at /workspace/speech2face."
    )
    if not MIMI_WEIGHT_R:
        print(f"[download] Mimi (tokenizer-e351c8d8) from {PERSONAPLEX_REPO} ...")
        MIMI_WEIGHT_R = Path(hf_hub_download(
            repo_id=PERSONAPLEX_REPO, revision=PERSONAPLEX_REV,
            filename="tokenizer-e351c8d8-checkpoint125.safetensors", token=token))
    if not MOSHI_WEIGHT_R:
        print(f"[download] LM (model.safetensors, ~15 GB) from {PERSONAPLEX_REPO} ...")
        MOSHI_WEIGHT_R = Path(hf_hub_download(
            repo_id=PERSONAPLEX_REPO, revision=PERSONAPLEX_REV,
            filename="model.safetensors", token=token))

MIMI_WEIGHT, MOSHI_WEIGHT = Path(MIMI_WEIGHT_R), Path(MOSHI_WEIGHT_R)
assert MIMI_WEIGHT.exists() and MOSHI_WEIGHT.exists()
print(f"[ready] MIMI_WEIGHT  = {MIMI_WEIGHT}")
print(f"[ready] MOSHI_WEIGHT = {MOSHI_WEIGHT}")

# --- 3. test audio (audio_3.wav) ----------------------------------------------
TEST_FILE_URLS = {
    "audio_3.wav": "https://raw.githubusercontent.com/MoshiHead/n_IMT_s_Perso_v2_2_aug_compress_log_Emotion_anti_v1/main/IMTalker/assets/audio_3.wav",
}

def _resolve_test_audio(configured):
    candidates = []
    if configured:
        candidates.append(Path(configured))
    candidates += [Path.cwd() / "audio_3.wav", Path("/workspace") / "audio_3.wav",
                   PROJECT_ROOT / "audio_3.wav", REPO_DIR / "audio_3.wav"]
    for c in candidates:
        if c.exists() and c.stat().st_size > 0:
            return c
    dest = PROJECT_ROOT / "audio_3.wav"
    print("[download] audio_3.wav from the old pipeline repo ...")
    urllib.request.urlretrieve(TEST_FILE_URLS["audio_3.wav"], dest)
    return dest

TEST_AUDIO = _resolve_test_audio(TEST_AUDIO)
TEST_HELIUM = Path(TEST_HELIUM) if TEST_HELIUM else (EXP_DIR / "test_helium" / "audio_3.npy")
print(f"[test] audio: {TEST_AUDIO} ({TEST_AUDIO.stat().st_size / 1e6:.2f} MB)")
print(f"[test] helium (to be extracted): {TEST_HELIUM}")

# --- 4. summary ------------------------------------------------------------------
print("\nCheckpoint summary:")
print(f"  base student:   {STUDENT_CKPT} (exists={STUDENT_CKPT.exists()})")
print(f"  renderer:       {RENDERER_CKPT} (exists={RENDERER_CKPT.exists()})")
print(f"  mimi:           {MIMI_WEIGHT}")
print(f"  LM:             {MOSHI_WEIGHT}")
assert STUDENT_CKPT.exists(), f"student checkpoint missing: {STUDENT_CKPT}"
assert RENDERER_CKPT.exists(), f"renderer checkpoint missing: {RENDERER_CKPT}"
print("\n[OK] All checkpoints resolved.")


## Helium feature extraction

The student consumes **raw 4096-d Helium hidden states @ 12.5 Hz** -- the
same representation it sees at live time. The cell below computes them for
every dataset clip (and for the test audio), using the **exact recipe of the
live server's `HeliumExtractor._extract_raw`**:

```
wav @ 24 kHz -> Mimi encode -> first 8 codebooks
  -> sum_q LM.emb[q](codes[:, q]) + LM.text_emb(padding)
  -> LM transformer (batch mode) -> forward hook on layers[-2]
  -> [T, 4096] fp32
```

Notes:

* **Layer choice matches deployment.** The live direct-reply-hidden path
  captures the same layer (`capture_layer=-2`) of the same LM, so training
  features and live features are the same space.
* The dataset's 768-d wav2vec2 `audio/*.npy` files are a different space and
  are **not** used -- this is the one step the FM version did not need.
* Runs as a **subprocess**, so the ~14 GB bf16 LM is fully released before
  training starts. Idempotent: existing `helium/*.npy` are skipped.
* Expect roughly 0.2-1 s of GPU time per clip-second (A100). For ~2.4k MEAD
  clips of ~4 s: 30-90 min, once.
* `--quantize_4bit` (`HELIUM_QUANTIZE_4BIT = True`) fits the LM in ~8 GB VRAM.


In [ ]:
# ============================================================================
# Helium feature extraction (dataset + test audio) -- idempotent, subprocess
# ============================================================================
import os
import subprocess
import sys
from pathlib import Path

extract_script = STUDENT_SRC / "extract_helium.py"
env = dict(os.environ)
env["PYTHONPATH"] = os.pathsep.join(
    [str(SRC_DIR / "niloys"), str(MOSHI_PKG)] +
    ([env["PYTHONPATH"]] if env.get("PYTHONPATH") else []))

common = [
    "--mimi_weight", str(MIMI_WEIGHT),
    "--moshi_weight", str(MOSHI_WEIGHT),
] + (["--quantize_4bit"] if HELIUM_QUANTIZE_4BIT else [])

# --- 1. dataset ---------------------------------------------------------------
cmd = [sys.executable, str(extract_script),
       "--dataset_dir", str(EMOTION_DATASET_DIR),
       "--wav_dirs", WAV_DIRS,
       "--out_dir", str(HELIUM_DIR)] + common
print("$", " ".join(cmd), flush=True)
r = subprocess.run(cmd, env=env)
if r.returncode != 0:
    raise RuntimeError("Helium extraction failed -- see the log above "
                       "(raw wav files missing?)")

# --- 2. test audio --------------------------------------------------------------
if not TEST_HELIUM.exists():
    cmd2 = [sys.executable, str(extract_script),
            "--single_wav", str(TEST_AUDIO),
            "--single_out", str(TEST_HELIUM)] + common
    print("$", " ".join(cmd2), flush=True)
    r2 = subprocess.run(cmd2, env=env)
    if r2.returncode != 0:
        raise RuntimeError("test-audio Helium extraction failed")
else:
    print(f"[skip] test helium already present: {TEST_HELIUM}")

# --- 3. report -------------------------------------------------------------------
n_helium = len(list(HELIUM_DIR.glob("*.npy")))
n_motion = len(list((EMOTION_DATASET_DIR / "motion").glob("*.pt")))
print(f"\nhelium features: {n_helium} / {n_motion} motion clips")
assert n_helium >= max(1, int(0.9 * n_motion)), (
    f"helium features missing for {n_motion - n_helium} clips -- "
    "check the extraction log for failed files")
print(f"[OK] test helium: {TEST_HELIUM} (exists={TEST_HELIUM.exists()})")


In [ ]:
# ============================================================================
# PRE-FLIGHT CHECKS: dataset, alignment, stats compatibility, checkpoints
# ============================================================================
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import torch

for _p in (str(SRC_DIR / "niloys"), str(SRC_DIR / "niloys" / "student")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from student.emotion_adapter import EMOTION_NAMES, parse_emotion_from_stem
from student.model import CausalMotionStudent, StudentConfig

# 1) dataset structure + helium/motion alignment
motion_files = sorted((EMOTION_DATASET_DIR / "motion").glob("*.pt"))
helium_files = sorted(HELIUM_DIR.glob("*.npy"))
assert motion_files, f"no motion/*.pt under {EMOTION_DATASET_DIR / 'motion'}"
assert helium_files, f"no helium/*.npy under {HELIUM_DIR} -- run the extraction cell"
counts = {"motion": len(motion_files), "helium": len(helium_files)}
print(f"dataset files: {counts}")
assert counts["helium"] >= max(1, int(0.9 * counts["motion"])), "helium coverage < 90%"
assert counts["motion"] >= max(10, VAL_SPLIT + 1), \
    f"need at least {VAL_SPLIT + 10} clips, found {counts['motion']}"

sample_m = torch.load(str(motion_files[0]), map_location="cpu")
sample_h = np.load(str(HELIUM_DIR / f"{motion_files[0].stem}.npy"), mmap_mode="r")
print(f"sample {motion_files[0].stem}: motion {tuple(sample_m.shape)} "
      f"helium {tuple(sample_h.shape)}")
assert sample_m.ndim == 2 and sample_m.shape[1] == 32, \
    f"motion must be [T, 32], got {tuple(sample_m.shape)}"
assert sample_h.ndim == 2 and sample_h.shape[1] == 4096, \
    f"helium must be [T, 4096], got {tuple(sample_h.shape)}"
ratio = sample_m.shape[0] / max(sample_h.shape[0], 1)
print(f"alignment ratio T_motion / T_helium = {ratio:.2f} (expect ~= 2.0)")
assert 1.5 <= ratio <= 2.5, "motion/helium rate mismatch -- extraction problem?"

# 2) emotion label distribution
emotion_distribution = Counter(parse_emotion_from_stem(mp.stem) for mp in motion_files)
print("\nemotion distribution:")
for eid in range(6):
    cnt = emotion_distribution.get(eid, 0)
    bar = "#" * (cnt // 50) if cnt > 0 else "(missing)"
    print(f"  [{eid}] {EMOTION_NAMES[eid]:10s}: {cnt:5d}  {bar}")
assert sum(emotion_distribution.values()) > 0, "no valid emotion labels found"

# 3) base student checkpoint + stats compatibility with the dataset
pack = torch.load(str(STUDENT_CKPT), map_location="cpu", weights_only=False)
cfg = StudentConfig(**pack["cfg"])
print(f"\nbase student: d_model={cfg.d_model} layers={cfg.n_layers} "
      f"heads={cfg.n_heads} window={cfg.attn_window} lookahead={cfg.lookahead} "
      f"| teacher step={pack.get('step')}")
ev = pack.get("eval") or {}
if ev:
    print(f"  teacher eval: tf_nmse={ev.get('tf_nmse'):.4f} free_nmse={ev.get('free_nmse'):.4f}")
assert cfg.helium_dim == 4096 and cfg.motion_dim == 32

# motion stats: dataset (sampled) vs the student's normalisation buffers
mm_ck, ms_ck = pack["model"]["motion_mean"], pack["model"]["motion_std"]
sample_files = motion_files[:: max(1, len(motion_files) // 60)]
vals = [torch.load(str(f), map_location="cpu").float() for f in sample_files]
vals = torch.cat(vals, 0)
mm_ds, ms_ds = vals.mean(0), vals.std(0).clamp_min(1e-5)
rel_mean = ((mm_ds - mm_ck).norm() / mm_ck.norm()).item()
rel_std = ((ms_ds - ms_ck).norm() / ms_ck.norm()).item()
print(f"\nmotion-space compatibility (dataset vs student stats, {len(sample_files)} clips):")
print(f"  mean: rel L2 = {rel_mean:.4f}")
print(f"  std : rel L2 = {rel_std:.4f}")
if rel_mean > 1.0 or rel_std > 1.0:
    raise RuntimeError(
        "The emotion dataset motion is in a DIFFERENT space than the student's\n"
        "32-dim motion (stats mismatch). Check how the dataset motion was extracted.")
elif rel_mean > 0.25 or rel_std > 0.25:
    print("  [warn] stats differ noticeably -- the adapter will have to bridge a\n"
          "         distribution gap. Training still works; watch the first test renders.")
else:
    print("  [OK] same motion space")

# 4) renderer checkpoint
rpack = torch.load(str(RENDERER_CKPT), map_location="cpu", weights_only=False)
assert "base_frame" in rpack["model"], "renderer has no embedded base_frame"
print(f"\nrenderer_narrow: res={rpack['cfg'].get('res')} "
      f"blink_len={rpack['cfg'].get('blink_len')} val_psnr={rpack.get('val_psnr')}")
print("  embedded base_frame: [OK] (no reference image needed for test renders)")

# 5) test assets
assert Path(TEST_AUDIO).exists(), f"test audio missing: {TEST_AUDIO}"
assert Path(TEST_HELIUM).exists(), f"test helium missing: {TEST_HELIUM} -- run extraction"
th = np.load(str(TEST_HELIUM), mmap_mode="r")
assert th.ndim == 2 and th.shape[1] == 4096
print(f"\ntest audio: {TEST_AUDIO}")
print(f"test helium: {tuple(th.shape)} ({th.shape[0] / 12.5:.1f}s)")

print("\nAll pre-flight checks passed -- ready to train.")


## Launch training

**Option A (recommended)** runs `train_emotion_student.py` as a background
process (robust against kernel restarts). Use the MONITOR cell to follow
progress.

**Option B** runs the same script in-notebook (blocks the kernel).

The trainer:

1. loads the frozen base student twice -- once as the trainable model, once
   as the **null anchor** (the frozen reference for the emotion-off path),
2. applies the emotion LoRA setup (~1.6M trainable params: conditioner +
   LoRA on `attn.qkv/proj`, MLPs, `head_abs/head_res`),
3. trains with AdamW (2e-4, cosine schedule + warmup) + EMA (0.999):
   * emotional samples (90%) -> MSE vs the dataset's emotional motion,
   * null samples (10%) -> MSE vs the **frozen base student's own output**
     on the same input (null-anchor distillation) -- this is what keeps
     "emotion off" identical to the current live behaviour,
   * + motion-velocity smoothness loss (0.1),
4. validates each epoch (emotional MSE + null-vs-base drift),
5. saves `checkpoints/last_train.pt` (resume) + periodic step checkpoints +
   the deployment pack `emotion_adapter.pt` (adapter-only, EMA weights),
6. renders test videos (base + happy + sad + angry) through the **live
   pipeline** (student step loop -> renderer_narrow) every **10 epochs**
   with the EMA weights -- training is never interrupted.

Auto-resume is on: re-running the launch cell continues from `last_train.pt`.


In [ ]:
# OPTION A -- Launch emotion LoRA training as a background process.
# Survives kernel restarts; monitor with the MONITOR cell below.
import os
import subprocess
import sys
from pathlib import Path

training_script = STUDENT_SRC / "train_emotion_student.py"
console_log = EXP_DIR / "train_console.log"
EXP_DIR.mkdir(parents=True, exist_ok=True)

cmd = [sys.executable, str(training_script),
    "--student_ckpt",        str(STUDENT_CKPT),
    "--dataset_dir",         str(EMOTION_DATASET_DIR),
    "--exp_root",            str(EXP_ROOT),
    "--exp_name",            EXP_NAME,
    "--lora_rank",           str(LORA_RANK),
    "--lora_alpha",          str(LORA_ALPHA),
    "--lora_dropout",        str(LORA_DROPOUT),
    "--lora_mlp",            "1" if LORA_MLP else "0",
    "--lora_heads",          "1" if LORA_HEADS else "0",
    "--steps_per_clip",      str(STEPS_PER_CLIP),
    "--ctx_noise",           str(CTX_NOISE),
    "--batch_size",          str(BATCH_SIZE),
    "--lr",                  str(LR),
    "--warmup",              str(WARMUP),
    "--max_steps",           str(MAX_STEPS),
    "--max_epochs",          str(MAX_EPOCHS),
    "--grad_clip",           str(GRAD_CLIP),
    "--ema_decay",           str(EMA_DECAY),
    "--null_drop",           str(NULL_DROP),
    "--anchor_w",            str(ANCHOR_W),
    "--vel_w",               str(VEL_W),
    "--save_every",          str(SAVE_FREQ),
    "--log_every",           str(LOG_EVERY),
    "--val_split",           str(VAL_SPLIT),
    "--num_workers",         str(NUM_WORKERS),
    "--seed",                str(SEED),
    "--test_every_n_epochs", str(TEST_EVERY_N),
    "--test_motion_gain",    str(TEST_MOTION_GAIN),
]
if not AUTO_RESUME:
    cmd.append("--no_auto_resume")
if EMOTION_FILTER:
    cmd += ["--emotion_filter"] + [str(x) for x in EMOTION_FILTER]
if TEST_EVERY_N and TEST_HELIUM and Path(TEST_HELIUM).exists():
    cmd += ["--test_helium", str(TEST_HELIUM),
            "--test_audio", str(TEST_AUDIO),
            "--renderer_ckpt", str(RENDERER_CKPT)]

env = dict(os.environ)
env["PYTHONPATH"] = os.pathsep.join(
    [str(SRC_DIR / "niloys"), str(MOSHI_PKG)] +
    ([env["PYTHONPATH"]] if env.get("PYTHONPATH") else []))

print("Launching:", " ".join(cmd[:8]), "...\n")
with open(console_log, "a") as logf:
    kwargs = dict(stdout=logf, stderr=subprocess.STDOUT, cwd=str(PROJECT_ROOT), env=env)
    if os.name == "posix":
        kwargs["start_new_session"] = True
    else:
        kwargs["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP
    proc = subprocess.Popen(cmd, **kwargs)

(EXP_DIR / "train.pid").write_text(str(proc.pid))
print(f"Training started  | pid={proc.pid}")
print(f"Console log:      {console_log}")
print(f"Checkpoint dir:   {EXP_DIR / 'checkpoints'}")
print(f"Deployment pack:  {EXP_DIR / 'emotion_adapter.pt'}")
print(f"Test videos:      {EXP_DIR / 'test_videos'} (every {TEST_EVERY_N} epochs)")
print("Run the MONITOR cell below to follow progress.")


In [ ]:
# OPTION B -- Run emotion LoRA training directly in this notebook.
# This blocks the kernel until training finishes.
import sys
from pathlib import Path

for _p in (str(SRC_DIR / "niloys"),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import student.train_emotion_student as train_mod

argv = [
    "--student_ckpt",        str(STUDENT_CKPT),
    "--dataset_dir",         str(EMOTION_DATASET_DIR),
    "--exp_root",            str(EXP_ROOT),
    "--exp_name",            EXP_NAME,
    "--lora_rank",           str(LORA_RANK),
    "--lora_alpha",          str(LORA_ALPHA),
    "--lora_dropout",        str(LORA_DROPOUT),
    "--lora_mlp",            "1" if LORA_MLP else "0",
    "--lora_heads",          "1" if LORA_HEADS else "0",
    "--steps_per_clip",      str(STEPS_PER_CLIP),
    "--ctx_noise",           str(CTX_NOISE),
    "--batch_size",          str(BATCH_SIZE),
    "--lr",                  str(LR),
    "--warmup",              str(WARMUP),
    "--max_steps",           str(MAX_STEPS),
    "--max_epochs",          str(MAX_EPOCHS),
    "--grad_clip",           str(GRAD_CLIP),
    "--ema_decay",           str(EMA_DECAY),
    "--null_drop",           str(NULL_DROP),
    "--anchor_w",            str(ANCHOR_W),
    "--vel_w",               str(VEL_W),
    "--save_every",          str(SAVE_FREQ),
    "--log_every",           str(LOG_EVERY),
    "--val_split",           str(VAL_SPLIT),
    "--num_workers",         str(NUM_WORKERS),
    "--seed",                str(SEED),
    "--test_every_n_epochs", str(TEST_EVERY_N),
    "--test_motion_gain",    str(TEST_MOTION_GAIN),
]
if not AUTO_RESUME:
    argv.append("--no_auto_resume")
if EMOTION_FILTER:
    argv += ["--emotion_filter"] + [str(x) for x in EMOTION_FILTER]
if TEST_EVERY_N and TEST_HELIUM and Path(TEST_HELIUM).exists():
    argv += ["--test_helium", str(TEST_HELIUM),
             "--test_audio", str(TEST_AUDIO),
             "--renderer_ckpt", str(RENDERER_CKPT)]

train_mod.main(argv)


In [ ]:
# MONITOR -- re-run this cell anytime to check training progress
import os
from pathlib import Path

def _tail(path, n=20):
    path = Path(path)
    if not path.exists():
        print(f"  (no {path.name} yet)")
        return
    lines = path.read_text(errors="replace").splitlines()
    print(f"--- Last {min(n, len(lines))} lines of {path.name} ---")
    print("\n".join(lines[-n:]))

pid_file = EXP_DIR / "train.pid"
if pid_file.exists():
    pid = int(pid_file.read_text().strip())
    try:
        os.kill(pid, 0)
        print(f"Process {pid} is RUNNING [OK]")
    except OSError:
        print(f"Process {pid} is NOT running (finished or stopped)")

ckpts = sorted((EXP_DIR / "checkpoints").glob("*.pt"))
print(f"Checkpoints saved: {len(ckpts)}")
for c in ckpts[-4:]:
    print(f"  {c.name} ({c.stat().st_size / 1e6:.1f} MB)")

pack = EXP_DIR / "emotion_adapter.pt"
print(f"Deployment pack: {'present' if pack.exists() else 'not yet'} {pack if pack.exists() else ''}")

vids = sorted((EXP_DIR / "test_videos").rglob("*.mp4"))
print(f"Test videos generated: {len(vids)}")
for v in vids[-4:]:
    print(f"  {v.parent.name}/{v.name} ({v.stat().st_size / 1e6:.1f} MB)")

print()
_tail(EXP_DIR / "train_log.txt", 15)
print()
_tail(EXP_DIR / "train_console.log", 15)


In [ ]:
# STOP training started with OPTION A (SIGTERM; the trainer saves and exits)
STOP_TRAINING = False

if not STOP_TRAINING:
    print("Set STOP_TRAINING = True and re-run this cell to stop background training.")
else:
    import os, signal
    pid_file = EXP_DIR / "train.pid"
    assert pid_file.exists(), "No train.pid found -- was training started with Option A?"
    pid = int(pid_file.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"Sent SIGTERM to process {pid}.")
        print("The trainer finishes the current step, saves last_train.pt + "
              "emotion_adapter.pt, then exits.")
    except ProcessLookupError:
        print(f"Process {pid} already stopped.")


## Validate the trained adapter

Three checks, then the full render:

1. **Contents** -- the deployment pack holds the emotion conditioner + LoRA
   keys (raw and EMA), the student's cfg and the LoRA cfg.
2. **Emotion geometry** -- the 6 emotion biases are distinct and non-zero
   (pairwise L2), and the **null bias is exactly zero**.
3. **Emotion-off fidelity** -- the null-conditioned student output vs the
   frozen base student on the test Helium: this is the number that tells you
   whether "emotion off" still behaves like the current live system.

Then render **base + all 6 emotions** through the live path (student step
loop -> renderer_narrow) and read the renderer-compatibility metrics.


In [ ]:
# Load the trained emotion adapter and verify its contents + emotion geometry
import sys
from pathlib import Path

import numpy as np
import torch

for _p in (str(SRC_DIR / "niloys"),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from student.emotion_adapter import EMOTION_NAMES, NULL_EMOTION_IDX
import student.test_emotions_student as ts

EVAL_CKPT = None  # None = auto-pick emotion_adapter.pt (EMA deployment pack)

if EVAL_CKPT is None:
    EVAL_CKPT = EXP_DIR / "emotion_adapter.pt"
    if not Path(EVAL_CKPT).exists():
        steps = sorted((EXP_DIR / "checkpoints").glob("step-*.pt"),
                       key=lambda p: p.stat().st_mtime)
        EVAL_CKPT = steps[-1] if steps else None
assert EVAL_CKPT and Path(EVAL_CKPT).exists(), \
    f"No trained adapter found in {EXP_DIR} -- train first"
print(f"Evaluating: {EVAL_CKPT}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, cfg = ts.load_emotion_student(str(STUDENT_CKPT), str(EVAL_CKPT), device)

# ---- 1. contents -------------------------------------------------------------
pack = torch.load(str(EVAL_CKPT), map_location="cpu", weights_only=False)
state = pack.get("ema_adapter") or pack["adapter"]
cond_keys = [k for k in state if "emotion_conditioner" in k]
lora_keys = [k for k in state if "lora_" in k]
print(f"\nadapter keys: {len(state)} total | conditioner: {len(cond_keys)} | LoRA: {len(lora_keys)}")
print(f"lora_cfg: {pack.get('lora_cfg')} | step: {pack.get('step')}")
proj_w = state.get("emotion_conditioner.proj.0.weight")
if proj_w is not None:
    print(f"conditioner first-layer weight norm: {proj_w.norm().item():.4f}")

# ---- 2. emotion geometry -------------------------------------------------------
print("\nemotion bias norms (d_model bias, constant over time):")
biases = {}
with torch.no_grad():
    for idx in range(6):
        b = model.emotion_conditioner(torch.tensor([idx], device=device), length=1)
        biases[EMOTION_NAMES[idx]] = b[0, 0].float().cpu()
        print(f"  [{idx}] {EMOTION_NAMES[idx]:10s}: norm={biases[EMOTION_NAMES[idx]].norm().item():.4f}")
    b_null = model.emotion_conditioner(
        torch.tensor([NULL_EMOTION_IDX], device=device), length=1)
print(f"  [6] null      : norm={b_null.norm().item():.6f} (must be exactly 0)")
assert float(b_null.norm()) == 0.0

print("\npairwise L2 distances between emotion biases:")
names = [EMOTION_NAMES[i] for i in range(6)]
dists = []
for i, n1 in enumerate(names):
    for j, n2 in enumerate(names):
        if j <= i:
            continue
        d = (biases[n1] - biases[n2]).norm().item()
        dists.append(d)
        print(f"  {n1:10s} vs {n2:10s}: {d:.4f}")
print(f"min pairwise distance: {min(dists):.4f}")

# ---- 3. emotion-off fidelity -----------------------------------------------------
helium = torch.from_numpy(np.load(str(TEST_HELIUM))).float()
m_null = ts.sample_motion(model, cfg, helium, NULL_EMOTION_IDX, motion_gain=1.0)
# base student reference: reload WITHOUT the adapter
base_model, _ = ts.load_emotion_student(str(STUDENT_CKPT), None, device, verbose=False)
m_ref = ts.sample_motion(base_model, cfg, helium, NULL_EMOTION_IDX, motion_gain=1.0)
drift = (m_null - m_ref).norm(dim=-1).mean().item()
scale = m_ref.norm(dim=-1).mean().item()
print(f"\nemotion-off fidelity on the test helium:")
print(f"  null-output vs base-student mean L2 = {drift:.5f}")
print(f"  (base motion mean frame norm = {scale:.3f}; relative drift = {drift / max(scale, 1e-6):.4f})")
del base_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("\n[OK] adapter verified.")


In [ ]:
# ============================================================================
# Render test videos: BASE (emotion off) + ALL 6 EMOTIONS through the live path
# Outputs to: {EXP_DIR}/final_emotion_tests/
#   base.mp4, 0_neutral.mp4, 1_happy.mp4, ..., 5_fear.mp4 + metrics.json
# ============================================================================
import sys
from pathlib import Path

import numpy as np
import torch

for _p in (str(SRC_DIR / "niloys"),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import student.test_emotions_student as ts

test_out_dir = EXP_DIR / "final_emotion_tests"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model may already be loaded by the previous cell; reload cleanly if not
if "EVAL_CKPT" not in dir() or EVAL_CKPT is None:
    EVAL_CKPT = EXP_DIR / "emotion_adapter.pt"
    assert Path(EVAL_CKPT).exists(), f"no trained adapter at {EVAL_CKPT} -- train first"
if "model" not in dir() or model is None:
    model, cfg = ts.load_emotion_student(str(STUDENT_CKPT), str(EVAL_CKPT), device)

renderer = ts.load_renderer(str(RENDERER_CKPT), device)
helium = torch.from_numpy(np.load(str(TEST_HELIUM))).float()

results = ts.render_test_set(
    model, cfg, renderer, helium, test_out_dir,
    audio_path=str(TEST_AUDIO),
    emotions=(None, 0, 1, 2, 3, 4, 5),
    motion_gain=TEST_MOTION_GAIN,
)

import json
(test_out_dir / "metrics.json").write_text(json.dumps(results, indent=2))

# ---- renderer compatibility verdict ------------------------------------------
emo = {k: v for k, v in results.items() if k != "base" and "pixel_mad_vs_base" in v}
print("\n=== renderer compatibility ===")
ml2 = np.mean([v["motion_l2_vs_base"] for v in emo.values()])
pmad = np.mean([v["pixel_mad_vs_base"] for v in emo.values()])
print(f"mean motion_l2_vs_base = {ml2:.4f}  (raw motion delta vs emotion-off)")
print(f"mean pixel_mad_vs_base = {pmad:.3f} (0..255 pixel scale)")
for k, v in emo.items():
    print(f"  {k:12s} motion_l2={v['motion_l2_vs_base']:.4f} pixel_mad={v['pixel_mad_vs_base']:.3f}")
if pmad < 0.5:
    print("\n[WARN] the rendered pixel delta is very small: the distilled renderer\n"
          "       is not visualising the emotion motion delta. The emotion IS in\n"
          "       the motion (see motion_l2 above) -- the renderer itself needs\n"
          "       fine-tuning on emotional motion to show it (next step).")
else:
    print("\n[OK] the emotion delta is visible in the rendered pixels.")
print(f"\nvideos + metrics in: {test_out_dir}")


## Deploying into the live streaming pipeline

The deployment pack (`emotion_adapter.pt`, ~7 MB) loads on top of the
**unmodified** `motion_student_38M.pt`. The launcher written by this notebook
-- `src/niloys/live/imtalker_try19_student_live_emotion.py` -- is a drop-in
replacement for `imtalker_try19_student_live.py`: it wraps `load_student()`,
installs the adapter, and then runs try19 unchanged.

### Launch (same command as run_x86.sh, two changes)

```bash
# 1) point at the emotion launcher instead of imtalker_try19_student_live.py
# 2) add --emotion_ckpt (and optionally --emotion_file / --emotion)
exec "$PY" -u "$SRC/niloys/live/imtalker_try19_student_live_emotion.py" \
  --emotion_ckpt /workspace/speech2face_emotion/emotion_exps/emotion_student_v1/emotion_adapter.pt \
  --emotion_file /workspace/emotion.txt \
  --emotion none \
  ... (every other run_x86.sh argument unchanged) ...
```

### Runtime switching

```bash
echo happy > /workspace/emotion.txt    # 0 neutral 1 happy 2 sad 3 angry 4 surprise 5 fear
echo none  > /workspace/emotion.txt    # back to the original behaviour
```

* The switch is **CUDA-graph safe**: the bias lives in a persistent buffer
  mutated in place, so `TRY19_STUDENT_GRAPH=1` replays pick it up on the next
  80 ms step.
* The KV history keeps the previous emotion's context, so transitions take
  ~1-2 s (the attention window) -- smooth, no discontinuity.
* `motion_gain` (GAIN=1.2) applies to emotional deviations too, making
  emotions more visible, exactly like it amplifies speech motion.
* The speculative-rollback safety of try19 is untouched (the bias buffer is
  engine-level, not stream-state).

### Copy the adapter to the streaming pod

```bash
scp emotion_adapter.pt pod:/workspace/speech2face/weights/
```


## After Training -- Output Files

All outputs are in `{EXP_ROOT}/{EXP_NAME}/`
(e.g. `/workspace/speech2face_emotion/emotion_exps/emotion_student_v1/`):

| Path | Content |
|------|---------|
| `emotion_adapter.pt` | **Deployment pack**: adapter-only weights (conditioner + LoRA A/B), raw + EMA, cfg + lora_cfg (~7 MB) |
| `checkpoints/last_train.pt` | Latest training state (auto-resume source: optimizer, EMA, RNG) |
| `checkpoints/step-XXXXXXX.pt` | Periodic training checkpoints (every `SAVE_FREQ` steps) |
| `train_log.txt` | Human-readable step/loss/lr/val log |
| `train_console.log` | Full console output (Option A) |
| `test_videos/epoch_XXXX/*.mp4` | EMA test renders every N epochs (base/happy/sad/angry) |
| `final_emotion_tests/*.mp4` | Final base + 6-emotion renders through the live pipeline |
| `final_emotion_tests/metrics.json` | Renderer-compatibility metrics |
| `test_helium/audio_3.npy` | Helium features for the test audio |

## If the emotion doesn't show in the videos

The metrics separate two failure modes:

1. `motion_l2_vs_base` small -> the *student* didn't learn the emotion
   (train longer, raise LR / LoRA rank, or check the dataset emotion balance).
2. `motion_l2_vs_base` healthy but `pixel_mad_vs_base` tiny -> the *renderer*
   can't visualise the emotion delta (it was distilled on neutral/speech
   motion only). The motion IS emotional; the decoder needs a fine-tune pass
   on emotional motion to show it. The validation cell prints exactly which
   case you are in -- no guessing.

## Tuning knobs (config cell)

| Knob | Default | Effect |
|---|---|---|
| `LORA_RANK` / `LORA_ALPHA` | 16 / 32 | capacity/strength of the emotion pathway |
| `NULL_DROP` | 0.10 | how much "emotion-off" anchoring (raise if the base behaviour drifts) |
| `ANCHOR_W` | 1.0 | weight of the null-anchor loss |
| `VEL_W` | 0.1 | motion smoothness |
| `LR` / `MAX_STEPS` | 2e-4 / 20000 | schedule |
| `TEST_MOTION_GAIN` | 1.2 | matches the live GAIN; raise for stronger expression |

## Reproducibility

`SEED` fixes the data order, crops and null-drop sampling; `AUTO_RESUME`
continues from `last_train.pt`, so re-running the notebook never restarts
from scratch. Set `AUTO_RESUME = False` (or delete the exp dir) to retrain.
